# Mathematics for Data Science: Interview Essentials

> A comprehensive, textbook-style reference covering the mathematical foundations frequently tested in hard interviews at top product companies (Google, Meta, Apple, Netflix, Amazon, etc.)

---

## Why This Notebook?

Data science interviews at top companies test not just your ability to use `sklearn.fit()`, but your **deep understanding** of why algorithms work. Interviewers want to know:
- Can you derive the gradient update for logistic regression from scratch?
- Why does PCA maximize variance? What's the connection to eigenvalues?
- How would you design an A/B test and what's the math behind the sample size calculation?
- What happens to gradient descent when the loss surface is non-convex?

This notebook answers all of these — with rigorous math, intuitive explanations, and working code.

---

## Table of Contents

| Part | Topic | Key Interview Questions |
|------|-------|------------------------|
| 1 | **Linear Algebra** | PCA derivation, SVD applications, matrix rank |
| 2 | **Probability & Statistics** | Bayes' theorem, conditional probability, distributions |
| 3 | **Statistical Inference** | MLE/MAP, hypothesis testing, A/B testing |
| 4 | **Calculus & Optimization** | Gradient descent, backpropagation, convexity |
| 5 | **Information Theory** | Entropy, KL divergence, cross-entropy loss |
| 6 | **Sampling & Monte Carlo** | MCMC, bootstrap, rejection sampling |
| 7 | **Advanced Topics** | Bias-variance tradeoff, kernel trick, matrix calculus |

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, linalg
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
np.random.seed(42)

print("All imports ready. Let's dive into the math!")

---
# Part 1: Linear Algebra

Linear algebra is the **backbone** of machine learning. Every algorithm — from linear regression to deep neural networks — is fundamentally a sequence of linear transformations followed by nonlinearities.

---

## 1.1 Vectors, Dot Products, and Norms

### Vectors as Data

In data science, every data point is a vector. A customer with features (age, income, purchase_count) is:

$$\mathbf{x} = \begin{bmatrix} 35 \\ 75000 \\ 12 \end{bmatrix} \in \mathbb{R}^3$$

### Dot Product: The Fundamental Operation

The dot product of two vectors $$\mathbf{a}, \mathbf{b} \in \mathbb{R}^n$$ is:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = \|\mathbf{a}\| \|\mathbf{b}\| \cos\theta$$

**Why it matters in interviews:**
- Linear regression prediction: $$\hat{y} = \mathbf{w}^T \mathbf{x} + b$$
- Cosine similarity in recommendation systems: $$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$
- Neural network forward pass: every layer computes $$\mathbf{z} = W\mathbf{x} + \mathbf{b}$$

### Vector Norms

| Norm | Formula | Use Case |
|------|---------|----------|
| $$L_1$$ (Manhattan) | $$\|\mathbf{x}\|_1 = \sum_i |x_i|$$ | Lasso regularization, sparse solutions |
| $$L_2$$ (Euclidean) | $$\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2}$$ | Ridge regularization, distance metrics |
| $$L_\infty$$ (Max) | $$\|\mathbf{x}\|_\infty = \max_i |x_i|$$ | Adversarial robustness |

### Real-World Example: Why L1 Gives Sparsity

Consider feature selection in a model predicting house prices. With 1000 features, most are irrelevant. L1 regularization (Lasso) adds $$\lambda \|\mathbf{w}\|_1$$ to the loss. Geometrically, the L1 ball has **corners** on the axes, and the optimal solution is more likely to land on a corner (where some $$w_i = 0$$), effectively removing features.

In [0]:
# === VECTORS, NORMS, AND COSINE SIMILARITY ===
# Real-world example: Recommendation system using user preference vectors

# User preference vectors (ratings for: Action, Comedy, Drama, Sci-Fi, Romance)
user_alice = np.array([5, 3, 1, 4, 2])  # Loves action & sci-fi
user_bob   = np.array([4, 4, 2, 5, 1])  # Similar to Alice
user_carol = np.array([1, 2, 5, 1, 5])  # Loves drama & romance

# --- Norms ---
print("="*60)
print("VECTOR NORMS")
print("="*60)
for name, vec in [("Alice", user_alice), ("Bob", user_bob), ("Carol", user_carol)]:
    print(f"\n{name}: {vec}")
    print(f"  L1 norm (sum of absolutes): {np.linalg.norm(vec, 1):.2f}")
    print(f"  L2 norm (Euclidean length):  {np.linalg.norm(vec, 2):.2f}")
    print(f"  L∞ norm (max element):       {np.linalg.norm(vec, np.inf):.2f}")

# --- Cosine Similarity ---
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\n" + "="*60)
print("COSINE SIMILARITY (used in recommendation engines)")
print("="*60)
print(f"\nAlice vs Bob (similar tastes):   {cosine_similarity(user_alice, user_bob):.4f}")
print(f"Alice vs Carol (different tastes): {cosine_similarity(user_alice, user_carol):.4f}")
print(f"Bob vs Carol:                      {cosine_similarity(user_bob, user_carol):.4f}")

# --- Geometric interpretation ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# L1 vs L2 balls (why L1 gives sparsity)
theta = np.linspace(0, 2*np.pi, 1000)

# L2 ball (circle)
axes[0].plot(np.cos(theta), np.sin(theta), 'b-', linewidth=2, label='$L_2$ ball (circle)')
# L1 ball (diamond)
l1_x = np.concatenate([np.linspace(0,1,250), np.linspace(1,0,250), 
                       np.linspace(0,-1,250), np.linspace(-1,0,250)])
l1_y = np.concatenate([np.linspace(1,0,250), np.linspace(0,-1,250),
                       np.linspace(-1,0,250), np.linspace(0,1,250)])
axes[0].plot(l1_x, l1_y, 'r-', linewidth=2, label='$L_1$ ball (diamond)')
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].axvline(0, color='gray', linewidth=0.5)
axes[0].scatter([1, -1, 0, 0], [0, 0, 1, -1], color='red', s=80, zorder=5)
axes[0].set_title('Why $L_1$ Regularization Gives Sparse Solutions\n(Corners touch axes → zero coefficients)')
axes[0].set_xlabel('$w_1$')
axes[0].set_ylabel('$w_2$')
axes[0].legend(fontsize=11)
axes[0].set_aspect('equal')

# Cosine similarity visualization
axes[1].quiver(0, 0, user_alice[0], user_alice[3], angles='xy', scale_units='xy', scale=1, color='blue', label='Alice')
axes[1].quiver(0, 0, user_bob[0], user_bob[3], angles='xy', scale_units='xy', scale=1, color='green', label='Bob')
axes[1].quiver(0, 0, user_carol[0], user_carol[3], angles='xy', scale_units='xy', scale=1, color='red', label='Carol')
axes[1].set_xlim(-1, 6)
axes[1].set_ylim(-1, 6)
axes[1].set_xlabel('Action Rating')
axes[1].set_ylabel('Sci-Fi Rating')
axes[1].set_title('User Preference Vectors\n(Angle = Taste Similarity)')
axes[1].legend()
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n✨ Key Insight: Cosine similarity measures DIRECTION (taste pattern),")
print("   not magnitude (how much they rate). Two generous raters with same")
print("   taste pattern get similarity ≈ 1.0.")

## 1.2 Matrices as Linear Transformations

### The Key Insight

A matrix $$A \in \mathbb{R}^{m \times n}$$ is not just a grid of numbers — it's a **function** that maps vectors from $$\mathbb{R}^n$$ to $$\mathbb{R}^m$$.

$$T(\mathbf{x}) = A\mathbf{x}$$

Every linear transformation (rotation, scaling, shearing, projection) can be represented as a matrix multiplication.

### Matrix Properties Critical for Interviews

**Rank:** The number of linearly independent rows (or columns).
- $$\text{rank}(A) = \dim(\text{column space of } A)$$
- A system $$A\mathbf{x} = \mathbf{b}$$ has a unique solution iff $$\text{rank}(A) = n$$ (full column rank)
- **Interview question:** "Your feature matrix has rank 5 but 100 columns. What does this mean?" → Only 5 features are truly independent; 95 are linear combinations of others (multicollinearity).

**Determinant:** Measures how the matrix scales volume.
- $$\det(A) = 0$$ → matrix is singular (not invertible), columns are linearly dependent
- $$|\det(A)|$$ = factor by which areas/volumes scale under the transformation

**Positive Definite Matrices:** A symmetric matrix $$A$$ is positive definite if:
$$\mathbf{x}^T A \mathbf{x} > 0 \quad \forall \mathbf{x} \neq \mathbf{0}$$

- The covariance matrix $$\Sigma$$ is always positive semi-definite
- The Hessian being positive definite at a point guarantees a local minimum
- Kernel matrices (Gram matrices) must be positive semi-definite

### Matrix Decompositions: The Power Tools

| Decomposition | Form | Use Case |
|---------------|------|----------|
| Eigendecomposition | $$A = Q\Lambda Q^{-1}$$ | PCA, spectral clustering |
| SVD | $$A = U\Sigma V^T$$ | Dimensionality reduction, recommenders |
| Cholesky | $$A = LL^T$$ | Efficient sampling from multivariate Gaussian |
| QR | $$A = QR$$ | Solving least squares, numerical stability |
| LU | $$A = LU$$ | Solving linear systems efficiently |

In [0]:
# === MATRIX RANK AND LINEAR DEPENDENCE ===
# Real-world: Detecting multicollinearity in feature engineering

print("="*60)
print("MATRIX RANK: DETECTING REDUNDANT FEATURES")
print("="*60)

# Simulate a dataset where some features are linearly dependent
np.random.seed(42)
n_samples = 100

# Independent features
age = np.random.normal(35, 10, n_samples)
income_monthly = np.random.normal(5000, 1500, n_samples)
years_experience = np.random.normal(10, 5, n_samples)

# Derived (linearly dependent) features
income_annual = 12 * income_monthly  # Perfect linear dependence!
income_biweekly = income_monthly / 2  # Another dependent feature
age_months = age * 12  # Yet another

# Create feature matrix
X_redundant = np.column_stack([age, income_monthly, years_experience, 
                                income_annual, income_biweekly, age_months])
feature_names = ['age', 'income_monthly', 'years_exp', 
                 'income_annual', 'income_biweekly', 'age_months']

print(f"\nFeature matrix shape: {X_redundant.shape}")
print(f"Matrix rank: {np.linalg.matrix_rank(X_redundant)}")
print(f"\n⚠️  6 features but rank 3 = only 3 independent features!")
print("   income_annual = 12 × income_monthly")
print("   income_biweekly = 0.5 × income_monthly")
print("   age_months = 12 × age")

# Now with just independent features
X_clean = np.column_stack([age, income_monthly, years_experience])
print(f"\nClean feature matrix shape: {X_clean.shape}")
print(f"Clean matrix rank: {np.linalg.matrix_rank(X_clean)}")
print("✅ Full rank = all features are independent")

# --- Condition Number: Numerical Stability ---
print("\n" + "="*60)
print("CONDITION NUMBER: NUMERICAL STABILITY")
print("="*60)
print(f"\nCondition number (redundant): {np.linalg.cond(X_redundant):.2e}")
print(f"Condition number (clean):     {np.linalg.cond(X_clean):.2e}")
print("\n💡 High condition number → matrix is 'almost singular'")
print("   Small changes in input cause large changes in output.")
print("   This is why multicollinearity breaks linear regression!")

# --- Determinant and Invertibility ---
print("\n" + "="*60)
print("POSITIVE DEFINITE MATRICES")
print("="*60)

# Covariance matrix is always positive semi-definite
cov_matrix = np.cov(X_clean.T)
eigenvalues = np.linalg.eigvalsh(cov_matrix)
print(f"\nCovariance matrix eigenvalues: {eigenvalues}")
print(f"All eigenvalues ≥ 0? {all(eigenvalues >= -1e-10)}")
print("✅ Covariance matrix is positive semi-definite (as expected)")
print("\n💡 Interview insight: If asked 'prove the covariance matrix is PSD':")
print("   For any vector z: zᵀΣz = zᵀE[(X-μ)(X-μ)ᵀ]z = E[(zᵀ(X-μ))²] ≥ 0")

## 1.3 Eigendecomposition: The Heartbeat of PCA

### Definition

An eigenvector $$\mathbf{v}$$ of matrix $$A$$ is a non-zero vector that only gets **scaled** (not rotated) when $$A$$ acts on it:

$$A\mathbf{v} = \lambda \mathbf{v}$$

where $$\lambda$$ is the eigenvalue (the scaling factor).

### Geometric Intuition

Imagine stretching a rubber sheet. Most points move in complicated ways, but some directions only stretch or compress. These are the eigenvectors — the **natural axes** of the transformation.

### Eigendecomposition

For a square matrix $$A \in \mathbb{R}^{n \times n}$$ with $$n$$ linearly independent eigenvectors:

$$A = Q \Lambda Q^{-1}$$

where:
- $$Q = [\mathbf{v}_1 | \mathbf{v}_2 | \ldots | \mathbf{v}_n]$$ (eigenvectors as columns)
- $$\Lambda = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_n)$$

For **symmetric matrices** (like covariance matrices): $$Q$$ is orthogonal ($$Q^{-1} = Q^T$$), so:

$$A = Q \Lambda Q^T$$

### Why Eigenvalues Matter for Data Science

1. **PCA**: Eigenvectors of the covariance matrix = principal components. Eigenvalues = variance explained.
2. **Graph algorithms**: Eigenvectors of the Laplacian matrix → spectral clustering.
3. **Stability analysis**: Eigenvalues of the Jacobian determine if a system is stable.
4. **PageRank**: The dominant eigenvector of the web graph's transition matrix.

### The Spectral Theorem (Interview Gold)

For any real symmetric matrix $$A$$:
1. All eigenvalues are **real**
2. Eigenvectors corresponding to distinct eigenvalues are **orthogonal**
3. $$A$$ can always be diagonalized: $$A = Q\Lambda Q^T$$

This is why PCA always works on covariance matrices!

In [0]:
# === EIGENDECOMPOSITION AND PCA FROM SCRATCH ===
# Real-world: Compressing high-dimensional customer data

print("="*60)
print("EIGENDECOMPOSITION: PCA FROM FIRST PRINCIPLES")
print("="*60)

# Generate correlated 2D data (height vs arm span - naturally correlated)
np.random.seed(42)
n = 200
# True underlying structure: one principal direction
height = np.random.normal(170, 10, n)  # cm
arm_span = 0.95 * height + np.random.normal(0, 3, n)  # Strongly correlated

X = np.column_stack([height, arm_span])

# Step 1: Center the data (subtract mean)
X_centered = X - X.mean(axis=0)
print(f"\nData shape: {X.shape}")
print(f"Mean (before centering): {X.mean(axis=0)}")
print(f"Mean (after centering):  {X_centered.mean(axis=0).round(10)}")

# Step 2: Compute covariance matrix
# Cov = (1/(n-1)) * X_centered^T @ X_centered
cov_matrix = (X_centered.T @ X_centered) / (n - 1)
print(f"\nCovariance matrix:")
print(f"  [{cov_matrix[0,0]:.2f}  {cov_matrix[0,1]:.2f}]")
print(f"  [{cov_matrix[1,0]:.2f}  {cov_matrix[1,1]:.2f}]")
print(f"\nCorrelation: {cov_matrix[0,1] / np.sqrt(cov_matrix[0,0]*cov_matrix[1,1]):.4f}")

# Step 3: Eigendecomposition of covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)  # eigh for symmetric matrices

# Sort by decreasing eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):")
print(f"  PC1: [{eigenvectors[0,0]:.4f}, {eigenvectors[1,0]:.4f}]")
print(f"  PC2: [{eigenvectors[0,1]:.4f}, {eigenvectors[1,1]:.4f}]")

# Variance explained
var_explained = eigenvalues / eigenvalues.sum()
print(f"\nVariance explained:")
print(f"  PC1: {var_explained[0]*100:.1f}% (the 'height-arm_span' direction)")
print(f"  PC2: {var_explained[1]*100:.1f}% (perpendicular noise)")

# Step 4: Project data onto principal components
X_pca = X_centered @ eigenvectors

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original data with eigenvectors
axes[0].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, s=20)
mean = X_centered.mean(axis=0)
for i, (eigval, eigvec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    # Scale eigenvector by sqrt(eigenvalue) for visualization
    axes[0].annotate('', xy=mean + 3*np.sqrt(eigval)*eigvec, xytext=mean,
                    arrowprops=dict(arrowstyle='->', color=['red','blue'][i], lw=3))
axes[0].set_xlabel('Height (centered)')
axes[0].set_ylabel('Arm Span (centered)')
axes[0].set_title('Original Data + Eigenvectors\n(Red=PC1, Blue=PC2)')
axes[0].set_aspect('equal')

# Projected data
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=20, color='green')
axes[1].set_xlabel('PC1 (captures most variance)')
axes[1].set_ylabel('PC2 (residual noise)')
axes[1].set_title('Data in PCA Space\n(Decorrelated!)')
axes[1].set_aspect('equal')

# Scree plot
axes[2].bar([1, 2], var_explained * 100, color=['red', 'blue'], alpha=0.7)
axes[2].set_xlabel('Principal Component')
axes[2].set_ylabel('Variance Explained (%)')
axes[2].set_title('Scree Plot\n(How many components to keep?)')
axes[2].set_xticks([1, 2])

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("🎯 INTERVIEW KEY POINTS:")
print("="*60)
print("1. PCA finds directions of MAXIMUM VARIANCE")
print("2. These directions ARE the eigenvectors of the covariance matrix")
print("3. Eigenvalues TELL YOU how much variance each direction captures")
print("4. Projected data is DECORRELATED (covariance matrix becomes diagonal)")
print(f"\nVerification - Covariance of projected data:")
print(f"  Cov(PC1, PC2) = {np.cov(X_pca.T)[0,1]:.2e} ≈ 0 ✅")

## 1.4 Singular Value Decomposition (SVD)

### The Most Important Decomposition in Data Science

SVD generalizes eigendecomposition to **any** matrix (not just square ones). For $$A \in \mathbb{R}^{m \times n}$$:

$$A = U \Sigma V^T$$

where:
- $$U \in \mathbb{R}^{m \times m}$$: left singular vectors (orthogonal) — patterns in rows
- $$\Sigma \in \mathbb{R}^{m \times n}$$: diagonal matrix of singular values $$\sigma_1 \geq \sigma_2 \geq \ldots \geq 0$$
- $$V \in \mathbb{R}^{n \times n}$$: right singular vectors (orthogonal) — patterns in columns

### Connection to Eigendecomposition

$$A^T A = V \Sigma^T \Sigma V^T \quad \Rightarrow \quad \text{eigenvectors of } A^T A = V$$

$$A A^T = U \Sigma \Sigma^T U^T \quad \Rightarrow \quad \text{eigenvectors of } A A^T = U$$

$$\sigma_i = \sqrt{\lambda_i(A^T A)}$$

### Connection to PCA

For centered data matrix $$X \in \mathbb{R}^{n \times p}$$ (n samples, p features):

$$\text{Covariance} = \frac{1}{n-1} X^T X$$

If $$X = U\Sigma V^T$$, then:

$$\frac{1}{n-1} X^T X = V \frac{\Sigma^2}{n-1} V^T$$

So the **right singular vectors** $$V$$ are the principal components, and $$\frac{\sigma_i^2}{n-1}$$ are the eigenvalues of the covariance matrix.

### Low-Rank Approximation (The Eckart–Young Theorem)

The best rank-$$k$$ approximation of $$A$$ (in Frobenius norm) is:

$$A_k = U_k \Sigma_k V_k^T = \sum_{i=1}^{k} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$

This is the mathematical foundation of:
- **Image compression** (keep top-k singular values)
- **Latent Semantic Analysis** (topic modeling)
- **Collaborative filtering** (Netflix Prize approach)
- **Noise reduction** (small singular values = noise)

In [0]:
# === SVD: IMAGE COMPRESSION DEMO ===
# Real-world: How Netflix/Spotify compress user-item matrices

print("="*60)
print("SVD: LOW-RANK APPROXIMATION")
print("="*60)

# Create a synthetic "image" (or think of it as a user-item rating matrix)
# Simulate a 50x50 image with structure
x = np.linspace(0, 4*np.pi, 50)
y = np.linspace(0, 4*np.pi, 50)
X_grid, Y_grid = np.meshgrid(x, y)
image = np.sin(X_grid) * np.cos(Y_grid) + 0.5*np.sin(2*X_grid) + np.random.normal(0, 0.1, (50, 50))

# Perform SVD
U, S, Vt = np.linalg.svd(image, full_matrices=False)

print(f"Original matrix: {image.shape}")
print(f"U: {U.shape}, S: {S.shape}, V^T: {Vt.shape}")
print(f"\nSingular values (top 10): {S[:10].round(2)}")
print(f"Total 'energy': {(S**2).sum():.2f}")

# Reconstruct with different ranks
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

ranks = [1, 2, 5, 10, 15, 20, 30, 50]
for idx, k in enumerate(ranks):
    row, col = idx // 4, idx % 4
    # Rank-k approximation
    reconstruction = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    
    # Compression ratio
    original_params = 50 * 50  # 2500
    compressed_params = k * (50 + 50 + 1)  # U_k columns + V_k rows + k singular values
    ratio = original_params / compressed_params
    error = np.linalg.norm(image - reconstruction, 'fro') / np.linalg.norm(image, 'fro')
    
    axes[row, col].imshow(reconstruction, cmap='viridis')
    axes[row, col].set_title(f'Rank {k}\nCompression: {ratio:.1f}x\nError: {error:.1%}')
    axes[row, col].axis('off')

plt.suptitle('SVD Low-Rank Approximation\n(Same principle used in Netflix recommendations)', fontsize=14)
plt.tight_layout()
plt.show()

# --- Energy captured by top-k singular values ---
energy_cumulative = np.cumsum(S**2) / (S**2).sum()

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(range(1, len(S)+1), energy_cumulative * 100, 'bo-', markersize=8)
ax.axhline(95, color='red', linestyle='--', label='95% threshold')
ax.axhline(99, color='orange', linestyle='--', label='99% threshold')

# Find how many components for 95%
k_95 = np.argmax(energy_cumulative >= 0.95) + 1
k_99 = np.argmax(energy_cumulative >= 0.99) + 1
ax.axvline(k_95, color='red', linestyle=':', alpha=0.5)
ax.axvline(k_99, color='orange', linestyle=':', alpha=0.5)

ax.set_xlabel('Number of Singular Values (k)')
ax.set_ylabel('Cumulative Energy (%)')
ax.set_title('How Many Components Do We Need?')
ax.legend()
ax.annotate(f'k={k_95} for 95%', xy=(k_95, 95), fontsize=11, color='red')
ax.annotate(f'k={k_99} for 99%', xy=(k_99, 99), fontsize=11, color='orange')
plt.show()

print(f"\n💡 With just {k_95} components (out of 50), we capture 95% of the information!")
print(f"   Compression ratio: {50*50 / (k_95*(50+50+1)):.1f}x")
print(f"\n🎯 Interview application:")
print(f"   Netflix has a 100M users × 50K movies matrix (mostly empty).")
print(f"   SVD finds ~100 latent factors that explain user preferences.")
print(f"   Storage: 100M×50K → (100M + 50K) × 100 = massive compression!")

---
# Part 2: Probability & Statistics

Probability is the language of uncertainty. Every prediction a model makes is fundamentally a probabilistic statement.

---

## 2.1 Probability Foundations

### Axioms of Probability (Kolmogorov)

1. $$P(A) \geq 0$$ for any event $$A$$
2. $$P(\Omega) = 1$$ (something always happens)
3. For mutually exclusive events: $$P(A \cup B) = P(A) + P(B)$$

### Conditional Probability & Independence

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

Two events are independent iff $$P(A \cap B) = P(A) \cdot P(B)$$, which means $$P(A|B) = P(A)$$.

**Interview trap:** "Are uncorrelated variables independent?" → **No!** Uncorrelated means $$E[XY] = E[X]E[Y]$$ (no linear relationship). Independence means **no relationship at all** (including nonlinear). Example: $$X \sim N(0,1)$$ and $$Y = X^2$$ are uncorrelated but highly dependent!

---

## 2.2 Bayes' Theorem: The Foundation of Modern ML

$$P(\theta | D) = \frac{P(D | \theta) \cdot P(\theta)}{P(D)}$$

| Term | Name | Interpretation |
|------|------|----------------|
| $$P(\theta \mid D)$$ | Posterior | What we believe after seeing data |
| $$P(D \mid \theta)$$ | Likelihood | How well the model explains the data |
| $$P(\theta)$$ | Prior | What we believed before seeing data |
| $$P(D)$$ | Evidence | Normalizing constant |

### Real-World Example: Medical Testing

A disease affects 1% of the population. A test has:
- Sensitivity (true positive rate): 95%
- Specificity (true negative rate): 90%

**Question:** If you test positive, what's the probability you actually have the disease?

This is the famous base rate fallacy — most people guess ~90%, but the real answer is surprisingly low.

### Bayes in Machine Learning

- **Naïve Bayes classifier**: Assumes feature independence given class
- **Bayesian regression**: Puts priors on weights
- **MAP estimation**: Finding the mode of the posterior
- **Bayesian neural networks**: Uncertainty quantification

In [0]:
# === BAYES' THEOREM: THE BASE RATE FALLACY ===
# Real-world: Medical testing, fraud detection, spam filtering

print("="*60)
print("BAYES' THEOREM: MEDICAL TESTING EXAMPLE")
print("="*60)

# Parameters
prevalence = 0.01        # P(Disease) = 1%
sensitivity = 0.95       # P(Positive | Disease) = 95%
specificity = 0.90       # P(Negative | No Disease) = 90%

# Bayes' Theorem calculation
# P(Disease | Positive) = P(Positive | Disease) * P(Disease) / P(Positive)
# P(Positive) = P(Pos|Dis)*P(Dis) + P(Pos|NoDis)*P(NoDis)

p_positive = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
p_disease_given_positive = (sensitivity * prevalence) / p_positive

print(f"\nGiven:")
print(f"  P(Disease) = {prevalence:.1%} (prevalence)")
print(f"  P(Positive | Disease) = {sensitivity:.0%} (sensitivity)")
print(f"  P(Negative | No Disease) = {specificity:.0%} (specificity)")
print(f"\nCalculation:")
print(f"  P(Positive) = {sensitivity}*{prevalence} + {1-specificity}*{1-prevalence}")
print(f"             = {sensitivity*prevalence:.4f} + {(1-specificity)*(1-prevalence):.4f}")
print(f"             = {p_positive:.4f}")
print(f"\n  P(Disease | Positive) = ({sensitivity} × {prevalence}) / {p_positive:.4f}")
print(f"                        = {p_disease_given_positive:.4f}")
print(f"\n⚠️  RESULT: Only {p_disease_given_positive:.1%} chance of having the disease!")
print(f"   Most people intuitively guess ~90%. This is the BASE RATE FALLACY.")

# --- Visualize with a natural frequency tree ---
print("\n" + "="*60)
print("NATURAL FREQUENCY EXPLANATION (out of 10,000 people):")
print("="*60)

population = 10000
sick = int(population * prevalence)  # 100
healthy = population - sick  # 9900

true_positives = int(sick * sensitivity)  # 95
false_negatives = sick - true_positives   # 5
false_positives = int(healthy * (1 - specificity))  # 990
true_negatives = healthy - false_positives  # 8910

print(f"\n  Population: {population:,}")
print(f"  Actually sick: {sick} | Actually healthy: {healthy:,}")
print(f"  True Positives:  {true_positives} (sick, correctly detected)")
print(f"  False Positives: {false_positives} (healthy, incorrectly flagged)")
print(f"  Total Positives: {true_positives + false_positives}")
print(f"\n  P(Sick | Positive) = {true_positives} / {true_positives + false_positives} = {true_positives/(true_positives+false_positives):.4f}")

# --- How posterior changes with prevalence ---
prevalences = np.linspace(0.001, 0.5, 200)
posteriors = (sensitivity * prevalences) / (sensitivity * prevalences + (1-specificity) * (1-prevalences))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(prevalences * 100, posteriors * 100, 'b-', linewidth=2)
axes[0].axhline(50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
axes[0].axvline(1, color='green', linestyle=':', alpha=0.7, label='Our example (1%)')
axes[0].scatter([1], [p_disease_given_positive*100], color='red', s=100, zorder=5)
axes[0].set_xlabel('Disease Prevalence (%)')
axes[0].set_ylabel('P(Disease | Positive Test) %')
axes[0].set_title('Posterior Probability vs. Base Rate\n(Sensitivity=95%, Specificity=90%)')
axes[0].legend()

# Confusion matrix heatmap
cm = np.array([[true_positives, false_negatives], [false_positives, true_negatives]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
           xticklabels=['Predicted +', 'Predicted -'],
           yticklabels=['Actually +', 'Actually -'])
axes[1].set_title(f'Confusion Matrix (n={population:,})\nPrecision = {true_positives}/{true_positives+false_positives} = {true_positives/(true_positives+false_positives):.1%}')

plt.tight_layout()
plt.show()

print("\n🎯 Interview applications of Bayes:")
print("   - Fraud detection: 0.1% fraud rate → even good models have many false positives")
print("   - Spam filtering: P(spam | words) using Naïve Bayes")
print("   - A/B testing: Bayesian approach gives P(B > A | data) directly")

## 2.3 Probability Distributions Every Data Scientist Must Know

### Discrete Distributions

**Bernoulli** (single coin flip):
$$P(X = k) = p^k (1-p)^{1-k}, \quad k \in \{0, 1\}$$
$$E[X] = p, \quad \text{Var}(X) = p(1-p)$$

**Binomial** (n independent trials):
$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$
$$E[X] = np, \quad \text{Var}(X) = np(1-p)$$

*Use case: Number of users who click an ad out of n impressions.*

**Poisson** (rare events in a fixed interval):
$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$
$$E[X] = \lambda, \quad \text{Var}(X) = \lambda$$

*Use case: Number of server errors per hour, customer arrivals per minute.*

**Geometric** (trials until first success):
$$P(X = k) = (1-p)^{k-1}p$$
$$E[X] = \frac{1}{p}$$

*Use case: Expected number of interviews before getting an offer.*

### Continuous Distributions

**Normal (Gaussian):**
$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

*Why it's everywhere:* Central Limit Theorem! Sum of many independent effects → Gaussian.

**Exponential** (time between Poisson events):
$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$
$$E[X] = \frac{1}{\lambda}, \quad \text{Var}(X) = \frac{1}{\lambda^2}$$

*Memoryless property:* $$P(X > s + t | X > s) = P(X > t)$$

*Use case: Time between customer purchases, server request inter-arrival times.*

**Beta** (probability of probabilities):
$$f(p; \alpha, \beta) = \frac{p^{\alpha-1}(1-p)^{\beta-1}}{B(\alpha, \beta)}$$
$$E[X] = \frac{\alpha}{\alpha + \beta}$$

*Use case: Prior distribution for click-through rates in Bayesian A/B testing.*

### Key Relationships

$$\text{Binomial}(n, p) \xrightarrow{n \to \infty, np = \lambda} \text{Poisson}(\lambda)$$

$$\text{Binomial}(n, p) \xrightarrow{n \to \infty} \text{Normal}(np, np(1-p))$$

$$\text{Exponential}(\lambda) \sim \text{time between Poisson}(\lambda) \text{ events}$$

In [0]:
# === PROBABILITY DISTRIBUTIONS GALLERY ===
# With real-world data science applications

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Binomial: Ad clicks
n_impressions, click_rate = 1000, 0.03
x_binom = np.arange(0, 60)
axes[0, 0].bar(x_binom, stats.binom.pmf(x_binom, n_impressions, click_rate), 
               color='steelblue', alpha=0.7)
axes[0, 0].axvline(n_impressions * click_rate, color='red', linestyle='--', 
                   label=f'E[X] = np = {n_impressions*click_rate:.0f}')
axes[0, 0].set_title(f'Binomial(n={n_impressions}, p={click_rate})\nAd clicks out of {n_impressions} impressions')
axes[0, 0].set_xlabel('Number of clicks')
axes[0, 0].legend()

# 2. Poisson: Server errors
lambda_errors = 5
x_pois = np.arange(0, 20)
axes[0, 1].bar(x_pois, stats.poisson.pmf(x_pois, lambda_errors), 
               color='coral', alpha=0.7)
axes[0, 1].axvline(lambda_errors, color='red', linestyle='--', 
                   label=f'E[X] = λ = {lambda_errors}')
axes[0, 1].set_title(f'Poisson(λ={lambda_errors})\nServer errors per hour')
axes[0, 1].set_xlabel('Number of errors')
axes[0, 1].legend()

# 3. Geometric: Interviews until offer
p_offer = 0.15
x_geom = np.arange(1, 30)
axes[0, 2].bar(x_geom, stats.geom.pmf(x_geom, p_offer), 
               color='mediumseagreen', alpha=0.7)
axes[0, 2].axvline(1/p_offer, color='red', linestyle='--', 
                   label=f'E[X] = 1/p = {1/p_offer:.1f}')
axes[0, 2].set_title(f'Geometric(p={p_offer})\nInterviews until job offer')
axes[0, 2].set_xlabel('Number of interviews')
axes[0, 2].legend()

# 4. Normal: Heights
x_norm = np.linspace(140, 200, 200)
for mu, sigma, label in [(170, 7, 'Men'), (162, 6, 'Women')]:
    axes[1, 0].plot(x_norm, stats.norm.pdf(x_norm, mu, sigma), linewidth=2, label=f'{label} (μ={mu}, σ={sigma})')
axes[1, 0].fill_between(x_norm, stats.norm.pdf(x_norm, 170, 7), alpha=0.2)
axes[1, 0].set_title('Normal Distribution\nHuman heights (cm)')
axes[1, 0].set_xlabel('Height (cm)')
axes[1, 0].legend()

# 5. Exponential: Time between purchases
lambda_rate = 0.1  # 0.1 purchases per day = 1 every 10 days
x_exp = np.linspace(0, 50, 200)
axes[1, 1].plot(x_exp, stats.expon.pdf(x_exp, scale=1/lambda_rate), 
               color='purple', linewidth=2)
axes[1, 1].fill_between(x_exp, stats.expon.pdf(x_exp, scale=1/lambda_rate), alpha=0.2, color='purple')
axes[1, 1].axvline(1/lambda_rate, color='red', linestyle='--', 
                   label=f'E[X] = 1/λ = {1/lambda_rate:.0f} days')
axes[1, 1].set_title(f'Exponential(λ={lambda_rate})\nDays between purchases')
axes[1, 1].set_xlabel('Days')
axes[1, 1].legend()

# 6. Beta: Click-through rate prior
x_beta = np.linspace(0, 1, 200)
for a, b, label in [(2, 8, 'Weak prior: α=2,β=8'), 
                     (20, 80, 'Strong prior: α=20,β=80'),
                     (1, 1, 'Uninformative: α=1,β=1')]:
    axes[1, 2].plot(x_beta, stats.beta.pdf(x_beta, a, b), linewidth=2, label=label)
axes[1, 2].set_title('Beta Distribution\nPrior for click-through rate')
axes[1, 2].set_xlabel('CTR')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

print("\n🎯 Interview tip: 'When would you use Poisson vs Binomial?'")
print("   • Binomial: Fixed number of trials, each with same probability")
print("   • Poisson: Counting events in continuous time/space (no fixed n)")
print("   • Poisson ≈ Binomial when n is large and p is small (np = λ)")

## 2.4 Law of Large Numbers & Central Limit Theorem

### Law of Large Numbers (LLN)

The sample mean converges to the population mean as $$n \to \infty$$:

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \xrightarrow{P} \mu$$

**Why it matters:** This is why training on more data gives better estimates. It's the theoretical guarantee behind empirical averages.

### Central Limit Theorem (CLT) — The Most Important Theorem in Statistics

For i.i.d. random variables with mean $$\mu$$ and variance $$\sigma^2$$:

$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} N(0, 1) \quad \text{as } n \to \infty$$

Equivalently:
$$\bar{X}_n \approx N\left(\mu, \frac{\sigma^2}{n}\right)$$

### Why CLT is the Foundation of Everything

1. **Confidence intervals:** $$\bar{X} \pm z_{\alpha/2} \frac{\sigma}{\sqrt{n}}$$ works because CLT makes $$\bar{X}$$ normal
2. **Hypothesis testing:** t-tests and z-tests rely on CLT
3. **A/B testing:** We can test if two means differ because their difference is approximately normal
4. **Sample size formulas:** All derived from CLT

### Key Properties

- Works regardless of the original distribution (exponential, uniform, Poisson, etc.)
- Rate of convergence: $$O(1/\sqrt{n})$$
- Typically "kicks in" around $$n \geq 30$$ for symmetric distributions
- Slower for heavy-tailed or highly skewed distributions

### The $$\sqrt{n}$$ Effect (Interview Gold)

The standard error of the mean decreases as $$\frac{\sigma}{\sqrt{n}}$$:
- To halve the error, you need $$4\times$$ the data
- To reduce error by 10x, you need $$100\times$$ the data

This explains why:
- Diminishing returns in data collection
- A/B tests need surprisingly large samples
- Ensemble methods (averaging models) reduce variance by $$\frac{1}{\sqrt{n}}$$

In [0]:
# === CENTRAL LIMIT THEOREM: VISUAL PROOF ===
# Showing CLT works for ANY distribution

fig, axes = plt.subplots(3, 4, figsize=(18, 13))

# Three very non-normal distributions
distributions = [
    ('Exponential(λ=1)', lambda size: np.random.exponential(1, size)),
    ('Uniform(0,1)', lambda size: np.random.uniform(0, 1, size)),
    ('Bimodal (mixture)', lambda size: np.where(np.random.random(size) < 0.5, 
                                                  np.random.normal(-3, 0.5, size),
                                                  np.random.normal(3, 0.5, size)))
]

sample_sizes = [1, 5, 30, 500]
n_experiments = 10000

for row, (name, gen_func) in enumerate(distributions):
    for col, n in enumerate(sample_sizes):
        # Generate n_experiments sample means, each from n observations
        sample_means = np.array([gen_func(n).mean() for _ in range(n_experiments)])
        
        axes[row, col].hist(sample_means, bins=50, density=True, alpha=0.7, color='steelblue')
        
        # Overlay theoretical normal from CLT
        if n > 1:
            # Get true mean and std of original distribution
            big_sample = gen_func(100000)
            true_mu = big_sample.mean()
            true_sigma = big_sample.std()
            x_range = np.linspace(sample_means.min(), sample_means.max(), 100)
            clt_pdf = stats.norm.pdf(x_range, true_mu, true_sigma/np.sqrt(n))
            axes[row, col].plot(x_range, clt_pdf, 'r-', linewidth=2, label='CLT prediction')
        
        if row == 0:
            axes[row, col].set_title(f'n = {n}', fontsize=13, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=11)
        if n > 1:
            axes[row, col].legend(fontsize=9)

plt.suptitle('Central Limit Theorem: Distribution of Sample Means\n'
             '(Red = Normal prediction from CLT. Works for ANY original distribution!)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- The sqrt(n) effect ---
print("\n" + "="*60)
print("THE √n EFFECT: DIMINISHING RETURNS OF DATA")
print("="*60)

# Simulate standard errors for increasing sample sizes
ns = np.array([10, 50, 100, 500, 1000, 5000, 10000, 50000])
true_sigma = 10  # Say we're measuring conversion rate variance
standard_errors = true_sigma / np.sqrt(ns)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(ns, standard_errors, 'bo-', markersize=8, linewidth=2)
ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('Standard Error = σ/√n', fontsize=12)
ax.set_title('Diminishing Returns: Why You Need 4x Data to Halve the Error')
ax.set_xscale('log')

# Annotate key points
for n, se in zip(ns[::2], standard_errors[::2]):
    ax.annotate(f'n={n:,}\nSE={se:.2f}', xy=(n, se), 
               textcoords="offset points", xytext=(10, 10), fontsize=9)

plt.show()

print(f"\n💡 Practical implications:")
print(f"   n=100   → SE = {true_sigma/np.sqrt(100):.2f}")
print(f"   n=400   → SE = {true_sigma/np.sqrt(400):.2f} (4x data → half the error)")
print(f"   n=10000 → SE = {true_sigma/np.sqrt(10000):.2f} (100x data → 1/10th the error)")
print(f"\n🎯 This is why A/B tests at Google need millions of users:")
print(f"   To detect a 0.1% change with 95% confidence, you need enormous n.")

---
# Part 3: Statistical Inference

Statistical inference is about drawing conclusions from data — the bridge between probability theory and real-world decision making.

---

## 3.1 Maximum Likelihood Estimation (MLE)

### The Big Idea

Given observed data $$D = \{x_1, x_2, \ldots, x_n\}$$, find the parameter $$\theta$$ that makes this data **most probable**:

$$\hat{\theta}_{MLE} = \arg\max_{\theta} P(D | \theta) = \arg\max_{\theta} \prod_{i=1}^n P(x_i | \theta)$$

In practice, we maximize the **log-likelihood** (converts products to sums):

$$\hat{\theta}_{MLE} = \arg\max_{\theta} \sum_{i=1}^n \log P(x_i | \theta)$$

### Example: MLE for Gaussian

For $$X_i \sim N(\mu, \sigma^2)$$:

$$\ell(\mu, \sigma^2) = -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n(x_i - \mu)^2$$

Taking derivatives and setting to zero:
$$\hat{\mu}_{MLE} = \bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$$
$$\hat{\sigma}^2_{MLE} = \frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^2$$

**Interview note:** The MLE for variance divides by $$n$$, not $$n-1$$. The unbiased estimator uses $$n-1$$ (Bessel's correction). This is a common trick question!

### MLE = Minimizing Cross-Entropy = Minimizing KL Divergence

Maximizing log-likelihood is equivalent to minimizing:
$$-\frac{1}{n}\sum_{i=1}^n \log P(x_i | \theta)$$

which is the **cross-entropy** between the empirical distribution and the model. This connects MLE directly to the loss functions used in deep learning!

---

## 3.2 Maximum A Posteriori (MAP) Estimation

MAP adds a prior to MLE:

$$\hat{\theta}_{MAP} = \arg\max_{\theta} P(\theta | D) = \arg\max_{\theta} [\log P(D|\theta) + \log P(\theta)]$$

### The Regularization Connection (Interview Favorite!)

| Prior on weights | Regularization | Effect |
|-----------------|----------------|--------|
| $$P(w) \propto e^{-\lambda w^2}$$ (Gaussian prior) | L2 / Ridge | Shrinks weights toward zero |
| $$P(w) \propto e^{-\lambda |w|}$$ (Laplace prior) | L1 / Lasso | Sparsity (some weights = 0) |

Proof for L2: If $$P(w_j) = N(0, \tau^2)$$, then:
$$\log P(\theta) = -\frac{1}{2\tau^2}\sum_j w_j^2 + \text{const}$$

So MAP = MLE + $$\lambda \|\mathbf{w}\|_2^2$$ where $$\lambda = \frac{1}{2\tau^2}$$. **Regularization IS a Bayesian prior!**

In [0]:
# === MLE AND MAP ESTIMATION ===
# Real-world: Estimating conversion rate with limited data

print("="*60)
print("MLE vs MAP: ESTIMATING CONVERSION RATE")
print("="*60)

# Scenario: New product page, only 10 visitors, 3 converted
# What's the true conversion rate?

n_visitors = 10
n_conversions = 3

# --- MLE ---
# For Bernoulli: MLE = k/n
mle_estimate = n_conversions / n_visitors
print(f"\nScenario: {n_conversions} conversions out of {n_visitors} visitors")
print(f"\nMLE estimate: p̂ = {n_conversions}/{n_visitors} = {mle_estimate:.2f}")
print(f"  ⚠️  Problem: With so little data, MLE is very uncertain")
print(f"  ⚠️  Extreme case: 0/2 visitors → MLE says p=0 (clearly wrong!)")

# --- MAP with Beta prior ---
# Prior: Beta(alpha, beta) represents our belief before seeing data
# Posterior: Beta(alpha + k, beta + n - k)
alpha_prior, beta_prior = 5, 20  # Prior belief: ~20% conversion rate
alpha_post = alpha_prior + n_conversions
beta_post = beta_prior + (n_visitors - n_conversions)

map_estimate = (alpha_post - 1) / (alpha_post + beta_post - 2)
posterior_mean = alpha_post / (alpha_post + beta_post)

print(f"\nMAP estimate (Beta({alpha_prior},{beta_prior}) prior): {map_estimate:.4f}")
print(f"Posterior mean: {posterior_mean:.4f}")
print(f"  💡 MAP 'shrinks' the estimate toward the prior (regularization!)")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p_range = np.linspace(0, 1, 200)

# Plot prior, likelihood, posterior
prior = stats.beta.pdf(p_range, alpha_prior, beta_prior)
likelihood = stats.binom.pmf(n_conversions, n_visitors, p_range)
likelihood = likelihood / likelihood.max()  # Normalize for visualization
posterior = stats.beta.pdf(p_range, alpha_post, beta_post)

axes[0].plot(p_range, prior / prior.max(), 'b--', linewidth=2, label=f'Prior: Beta({alpha_prior},{beta_prior})')
axes[0].plot(p_range, likelihood, 'g-.', linewidth=2, label=f'Likelihood: Binom({n_conversions}/{n_visitors})')
axes[0].plot(p_range, posterior / posterior.max(), 'r-', linewidth=3, label=f'Posterior: Beta({alpha_post},{beta_post})')
axes[0].axvline(mle_estimate, color='green', linestyle=':', alpha=0.7, label=f'MLE = {mle_estimate:.2f}')
axes[0].axvline(map_estimate, color='red', linestyle=':', alpha=0.7, label=f'MAP = {map_estimate:.4f}')
axes[0].set_xlabel('Conversion Rate (p)')
axes[0].set_ylabel('Density (normalized)')
axes[0].set_title('Bayesian Updating: Prior × Likelihood ∝ Posterior')
axes[0].legend(fontsize=10)

# Show how MAP approaches MLE with more data
n_data_points = [5, 20, 50, 200, 1000]
true_p = 0.25

mle_estimates = []
map_estimates = []

for n in n_data_points:
    k = int(true_p * n)  # Observed successes
    mle_estimates.append(k / n)
    a_post = alpha_prior + k
    b_post = beta_prior + (n - k)
    map_estimates.append((a_post - 1) / (a_post + b_post - 2))

axes[1].plot(n_data_points, mle_estimates, 'go-', markersize=8, linewidth=2, label='MLE')
axes[1].plot(n_data_points, map_estimates, 'rs-', markersize=8, linewidth=2, label='MAP')
axes[1].axhline(true_p, color='black', linestyle='--', label=f'True p = {true_p}')
axes[1].set_xlabel('Number of observations')
axes[1].set_ylabel('Estimate of p')
axes[1].set_title('MAP → MLE as Data Increases\n(Prior becomes irrelevant with enough data)')
axes[1].legend()
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

print("\n🎯 Key insight: MAP = MLE + regularization")
print("   With little data: prior dominates (strong regularization)")
print("   With lots of data: likelihood dominates (MLE ≈ MAP)")
print("   This is EXACTLY what happens with L2 regularization strength!")

## 3.3 Hypothesis Testing & A/B Testing

### The Framework

1. **Null hypothesis** $$H_0$$: No effect (status quo)
2. **Alternative hypothesis** $$H_1$$: There IS an effect
3. Compute a **test statistic** from data
4. Calculate the **p-value**: probability of seeing this extreme a result if $$H_0$$ were true
5. **Reject** $$H_0$$ if p-value $$< \alpha$$ (significance level, typically 0.05)

### Types of Errors

| | $$H_0$$ True | $$H_0$$ False |
|---|---|---|
| **Reject $$H_0$$** | Type I Error ($$\alpha$$) | Correct! (Power = $$1-\beta$$) |
| **Fail to reject** | Correct! | Type II Error ($$\beta$$) |

### The Two-Sample Z-Test (A/B Testing)

For two groups with means $$\bar{X}_A, \bar{X}_B$$ and known/estimated variances:

$$Z = \frac{\bar{X}_B - \bar{X}_A}{\sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}}$$

Under $$H_0$$ (no difference), $$Z \sim N(0,1)$$ by CLT.

### Sample Size Formula (Derivation)

To detect effect size $$\delta$$ with power $$1-\beta$$ and significance $$\alpha$$:

$$n = \frac{(z_{\alpha/2} + z_\beta)^2 (\sigma_A^2 + \sigma_B^2)}{\delta^2}$$

For proportions (e.g., conversion rates $$p_A$$ vs $$p_B$$):

$$n = \frac{(z_{\alpha/2} + z_\beta)^2 [p_A(1-p_A) + p_B(1-p_B)]}{(p_B - p_A)^2}$$

### Common Interview Pitfalls

1. **Multiple testing**: Testing 20 metrics? Expected 1 false positive at $$\alpha=0.05$$ → Use Bonferroni correction
2. **Peeking**: Checking results daily inflates false positive rate → Use sequential testing
3. **Low power**: "We didn't find significance" ≠ "There's no effect". Maybe $$n$$ was too small!
4. **Practical vs. Statistical significance**: p < 0.001 with a 0.01% lift is statistically significant but useless

In [0]:
# === A/B TESTING: COMPLETE FRAMEWORK ===
# Real-world: Testing a new checkout button design at an e-commerce company

print("="*60)
print("A/B TEST: NEW CHECKOUT BUTTON DESIGN")
print("="*60)

# --- Step 1: Power Analysis (Before the test) ---
print("\n--- STEP 1: POWER ANALYSIS ---")

# Current conversion rate (control)
p_control = 0.12  # 12% conversion rate
# Minimum detectable effect (what's worth shipping?)
mde = 0.015  # Want to detect a 1.5 percentage point increase
p_treatment = p_control + mde

alpha = 0.05  # Significance level
power = 0.80  # Statistical power (1 - beta)

# Sample size formula for proportions
z_alpha = stats.norm.ppf(1 - alpha/2)  # 1.96
z_beta = stats.norm.ppf(power)          # 0.84

n_per_group = ((z_alpha + z_beta)**2 * (p_control*(1-p_control) + p_treatment*(1-p_treatment))) / mde**2
n_per_group = int(np.ceil(n_per_group))

print(f"\n  Baseline conversion: {p_control:.1%}")
print(f"  Minimum detectable effect: {mde:.1%} (relative: {mde/p_control:.1%})")
print(f"  Significance level (α): {alpha}")
print(f"  Power (1-β): {power}")
print(f"\n  → Required sample size per group: {n_per_group:,}")
print(f"  → Total sample needed: {2*n_per_group:,}")
print(f"  → At 10K visitors/day: ~{2*n_per_group/10000:.0f} days to run")

# --- Step 2: Simulate the experiment ---
print("\n--- STEP 2: RUN THE EXPERIMENT (simulated) ---")

np.random.seed(42)
true_p_control = 0.12
true_p_treatment = 0.135  # True effect is 1.5pp (exists!)

n = n_per_group
control_conversions = np.random.binomial(n, true_p_control)
treatment_conversions = np.random.binomial(n, true_p_treatment)

p_hat_c = control_conversions / n
p_hat_t = treatment_conversions / n

print(f"\n  Control:   {control_conversions}/{n} = {p_hat_c:.4f}")
print(f"  Treatment: {treatment_conversions}/{n} = {p_hat_t:.4f}")
print(f"  Observed lift: {p_hat_t - p_hat_c:.4f} ({(p_hat_t-p_hat_c)/p_hat_c:.2%} relative)")

# --- Step 3: Statistical test ---
print("\n--- STEP 3: STATISTICAL TEST ---")

# Pooled standard error (under H0: p_control = p_treatment)
p_pooled = (control_conversions + treatment_conversions) / (2 * n)
se = np.sqrt(p_pooled * (1 - p_pooled) * (2/n))

# Z-statistic
z_stat = (p_hat_t - p_hat_c) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # Two-tailed

print(f"\n  Pooled proportion: {p_pooled:.4f}")
print(f"  Standard error: {se:.5f}")
print(f"  Z-statistic: {z_stat:.4f}")
print(f"  p-value: {p_value:.6f}")

if p_value < alpha:
    print(f"\n  ✅ REJECT H₀: Significant at α={alpha}! The new design wins.")
else:
    print(f"\n  ❌ FAIL TO REJECT H₀: Not enough evidence at α={alpha}.")

# --- Step 4: Confidence Interval ---
se_unpooled = np.sqrt(p_hat_c*(1-p_hat_c)/n + p_hat_t*(1-p_hat_t)/n)
ci_lower = (p_hat_t - p_hat_c) - z_alpha * se_unpooled
ci_upper = (p_hat_t - p_hat_c) + z_alpha * se_unpooled
print(f"\n  95% CI for difference: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  (If CI doesn't contain 0, we reject H₀)")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Power curve
effect_sizes = np.linspace(0.001, 0.05, 100)
powers = []
for es in effect_sizes:
    # Power = P(reject H0 | H1 true)
    z_crit = z_alpha
    se_power = np.sqrt(2 * p_control * (1-p_control) / n)
    power_val = 1 - stats.norm.cdf(z_crit - es/se_power)
    powers.append(power_val)

axes[0].plot(effect_sizes * 100, powers, 'b-', linewidth=2)
axes[0].axhline(0.8, color='red', linestyle='--', label='80% power threshold')
axes[0].axvline(mde * 100, color='green', linestyle=':', label=f'MDE = {mde:.1%}')
axes[0].set_xlabel('True Effect Size (percentage points)')
axes[0].set_ylabel('Statistical Power')
axes[0].set_title(f'Power Curve (n={n:,} per group)')
axes[0].legend()

# Sampling distribution under H0 and H1
x_range = np.linspace(-4, 6, 200)
axes[1].plot(x_range, stats.norm.pdf(x_range, 0, 1), 'b-', linewidth=2, label='Under H₀ (no effect)')
axes[1].fill_between(x_range[x_range > 1.96], stats.norm.pdf(x_range[x_range > 1.96], 0, 1), 
                     alpha=0.3, color='red', label=f'Rejection region (α/2={alpha/2})')
axes[1].fill_between(x_range[x_range < -1.96], stats.norm.pdf(x_range[x_range < -1.96], 0, 1), 
                     alpha=0.3, color='red')

# Under H1
noncentrality = mde / se
axes[1].plot(x_range, stats.norm.pdf(x_range, noncentrality, 1), 'g-', linewidth=2, 
            label=f'Under H₁ (true effect={mde:.1%})')
axes[1].axvline(z_stat, color='purple', linewidth=2, linestyle='-', label=f'Observed Z={z_stat:.2f}')
axes[1].set_xlabel('Z-statistic')
axes[1].set_title('Null vs Alternative Distribution')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n🎯 Interview checklist for A/B testing:")
print("   1. Define metric and MDE BEFORE running the test")
print("   2. Calculate required sample size (power analysis)")
print("   3. Randomize properly (watch for selection bias)")
print("   4. Don't peek! (or use sequential testing)")
print("   5. Check for novelty effects and network effects")
print("   6. Multiple testing correction if >1 metric")

---
# Part 4: Calculus & Optimization

Optimization is **how machines learn**. Every training procedure is solving:
$$\hat{\theta} = \arg\min_{\theta} \mathcal{L}(\theta)$$

---

## 4.1 Derivatives, Gradients, and the Chain Rule

### Scalar Derivative
The derivative $$f'(x) = \frac{df}{dx}$$ gives the rate of change and the slope of the tangent line.

### Gradient (Multi-variable)
For $$f: \mathbb{R}^n \to \mathbb{R}$$, the gradient is the vector of partial derivatives:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$$

**Key property:** The gradient points in the direction of steepest ascent. Its magnitude tells you how steep.

### The Jacobian (Vector-valued functions)
For $$\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$$:

$$J = \begin{bmatrix} \frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n} \end{bmatrix}$$

### The Hessian (Second derivatives)
For $$f: \mathbb{R}^n \to \mathbb{R}$$:

$$H = \begin{bmatrix} \frac{\partial^2 f}{\partial x_1^2} & \frac{\partial^2 f}{\partial x_1 \partial x_2} & \cdots \\ \frac{\partial^2 f}{\partial x_2 \partial x_1} & \frac{\partial^2 f}{\partial x_2^2} & \cdots \\ \vdots & \vdots & \ddots \end{bmatrix}$$

- $$H$$ positive definite at point $$\mathbf{x}^*$$ → local minimum
- $$H$$ negative definite → local maximum
- $$H$$ indefinite (mixed eigenvalues) → saddle point

### The Chain Rule (Foundation of Backpropagation)

For composed functions $$f(g(x))$$:
$$\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$$

For neural networks with layers $$z_1 \to z_2 \to \ldots \to z_L \to \mathcal{L}$$:
$$\frac{\partial \mathcal{L}}{\partial W_1} = \frac{\partial \mathcal{L}}{\partial z_L} \cdot \frac{\partial z_L}{\partial z_{L-1}} \cdots \frac{\partial z_2}{\partial z_1} \cdot \frac{\partial z_1}{\partial W_1}$$

This is **backpropagation** — just the chain rule applied recursively!

In [0]:
# === GRADIENT DESCENT: FROM SCRATCH ===
# Real-world: Training a linear regression model

print("="*60)
print("GRADIENT DESCENT: LINEAR REGRESSION FROM FIRST PRINCIPLES")
print("="*60)

# Generate data: House prices (size -> price)
np.random.seed(42)
n_houses = 100
house_size = np.random.uniform(50, 200, n_houses)  # square meters
true_w, true_b = 3000, 50000  # ₹3000/sqm + ₹50K base
noise = np.random.normal(0, 20000, n_houses)
house_price = true_w * house_size + true_b + noise

# Normalize for better gradient descent behavior
X = (house_size - house_size.mean()) / house_size.std()
y = (house_price - house_price.mean()) / house_price.std()

print(f"Data: {n_houses} houses, predicting price from size")
print(f"True relationship: price = {true_w} × size + {true_b} + noise")

# --- Manual Gradient Descent ---
def compute_loss(w, b, X, y):
    """Mean Squared Error: L = (1/n) * sum((y - (wx + b))^2)"""
    predictions = w * X + b
    return np.mean((y - predictions) ** 2)

def compute_gradients(w, b, X, y):
    """Analytical gradients of MSE.
    dL/dw = -(2/n) * sum(x_i * (y_i - (wx_i + b)))
    dL/db = -(2/n) * sum(y_i - (wx_i + b))
    """
    n = len(X)
    predictions = w * X + b
    residuals = y - predictions
    dw = -(2/n) * np.sum(X * residuals)
    db = -(2/n) * np.sum(residuals)
    return dw, db

# Gradient descent with different learning rates
learning_rates = [0.001, 0.01, 0.1, 0.5]
n_iterations = 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, lr in enumerate(learning_rates):
    row, col = idx // 2, idx % 2
    
    # Initialize
    w, b = 0.0, 0.0
    losses = []
    w_history = [w]
    b_history = [b]
    
    for i in range(n_iterations):
        loss = compute_loss(w, b, X, y)
        losses.append(loss)
        
        dw, db = compute_gradients(w, b, X, y)
        w -= lr * dw
        b -= lr * db
        
        w_history.append(w)
        b_history.append(b)
    
    axes[row, col].plot(losses, linewidth=2)
    axes[row, col].set_xlabel('Iteration')
    axes[row, col].set_ylabel('MSE Loss')
    
    if losses[-1] < 100:  # Converged
        axes[row, col].set_title(f'LR = {lr} | Final loss: {losses[-1]:.4f}\nw={w:.4f}, b={b:.4f}', color='green')
    else:  # Diverged
        axes[row, col].set_title(f'LR = {lr} | DIVERGED!\nLoss exploding', color='red')
    axes[row, col].set_yscale('log' if max(losses) > 10 else 'linear')

plt.suptitle('Effect of Learning Rate on Gradient Descent Convergence', fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 Key observations:")
print("   - Too small LR (0.001): Converges but painfully slow")
print("   - Good LR (0.01-0.1): Converges quickly")
print("   - Too large LR (0.5): Oscillates or diverges!")
print("\n🎯 Interview question: 'What happens if learning rate is too high?'")
print("   Answer: Gradient updates overshoot the minimum, causing oscillation")
print("   or divergence. The loss INCREASES instead of decreasing.")

## 4.2 Convexity: When Optimization is "Easy"

### Definition

A function $$f$$ is **convex** if for all $$\mathbf{x}, \mathbf{y}$$ and $$\lambda \in [0, 1]$$:

$$f(\lambda \mathbf{x} + (1-\lambda)\mathbf{y}) \leq \lambda f(\mathbf{x}) + (1-\lambda) f(\mathbf{y})$$

Geometrically: the line segment between any two points on the graph lies **above** the function.

### Why Convexity Matters

- **Convex function**: Every local minimum is a global minimum. Gradient descent finds the optimal solution!
- **Non-convex function**: Multiple local minima, saddle points. No guarantee of finding global optimum.

### Which Loss Functions Are Convex?

| Loss | Convex? | Notes |
|------|---------|-------|
| MSE (linear regression) | ✅ Yes | Quadratic in weights |
| Logistic loss | ✅ Yes | Convex in linear weights |
| Cross-entropy (neural nets) | ❌ No | Non-convex due to nonlinear activations |
| SVM hinge loss | ✅ Yes | Piecewise linear = convex |

### Second-Order Condition

$$f$$ is convex iff the Hessian $$H$$ is positive semi-definite everywhere:
$$H = \nabla^2 f \succeq 0$$

### Gradient Descent Convergence Guarantees

For a convex function with L-Lipschitz continuous gradients:
- **Vanilla GD** converges at rate $$O(1/T)$$ (T = iterations)
- **Strongly convex** + L-smooth: $$O\left(\left(1 - \frac{\mu}{L}\right)^T\right)$$ (exponential!)
- The ratio $$\kappa = L/\mu$$ is the **condition number** — higher means slower convergence

### Practical Variants of Gradient Descent

| Method | Update Rule | Key Idea |
|--------|-------------|----------|
| SGD | $$\theta \leftarrow \theta - \eta \nabla_{\text{batch}} \mathcal{L}$$ | Use random subset (mini-batch) |
| Momentum | $$v \leftarrow \gamma v + \eta \nabla \mathcal{L}; \; \theta \leftarrow \theta - v$$ | Accumulate velocity |
| Adam | Adaptive LR + momentum | Per-parameter learning rates |

### Why Does SGD Work for Non-Convex Deep Learning?

1. **Saddle points** are more common than local minima in high dimensions
2. SGD noise helps escape shallow local minima
3. Overparameterized networks often have "flat" minima that generalize well
4. The loss landscape of practical neural networks is more benign than worst-case theory suggests

In [0]:
# === GRADIENT DESCENT VARIANTS: SGD vs MOMENTUM vs ADAM ===
# Real-world: Optimizing a non-convex loss surface

print("="*60)
print("OPTIMIZATION LANDSCAPE & GRADIENT DESCENT VARIANTS")
print("="*60)

# Define a 2D loss surface (Rosenbrock-like, non-convex)
def loss_surface(x, y):
    """A challenging optimization landscape with a narrow valley."""
    return (1 - x)**2 + 100*(y - x**2)**2

def grad_loss(x, y):
    """Gradient of Rosenbrock function."""
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return np.array([dx, dy])

# Implement optimizers
def sgd(grad_fn, x0, lr=0.0001, n_iter=5000):
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(n_iter):
        g = grad_fn(x[0], x[1])
        x = x - lr * g
        path.append(x.copy())
    return np.array(path)

def momentum(grad_fn, x0, lr=0.0001, gamma=0.9, n_iter=5000):
    path = [x0.copy()]
    x = x0.copy()
    v = np.zeros_like(x)
    for _ in range(n_iter):
        g = grad_fn(x[0], x[1])
        v = gamma * v + lr * g
        x = x - v
        path.append(x.copy())
    return np.array(path)

def adam(grad_fn, x0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, n_iter=5000):
    path = [x0.copy()]
    x = x0.copy()
    m = np.zeros_like(x)  # First moment
    v = np.zeros_like(x)  # Second moment
    for t in range(1, n_iter + 1):
        g = grad_fn(x[0], x[1])
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)  # Bias correction
        v_hat = v / (1 - beta2**t)
        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(x.copy())
    return np.array(path)

# Run all optimizers
x0 = np.array([-1.0, 1.0])
path_sgd = sgd(grad_loss, x0, lr=0.0001, n_iter=8000)
path_mom = momentum(grad_loss, x0, lr=0.0001, gamma=0.9, n_iter=8000)
path_adam = adam(grad_loss, x0, lr=0.01, n_iter=8000)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Contour plot with paths
x_grid = np.linspace(-2, 2, 200)
y_grid = np.linspace(-1, 3, 200)
X_g, Y_g = np.meshgrid(x_grid, y_grid)
Z = loss_surface(X_g, Y_g)

axes[0].contour(X_g, Y_g, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis', alpha=0.6)
axes[0].plot(path_sgd[:, 0], path_sgd[:, 1], 'r-', linewidth=1.5, alpha=0.7, label='Vanilla SGD')
axes[0].plot(path_mom[:, 0], path_mom[:, 1], 'b-', linewidth=1.5, alpha=0.7, label='Momentum')
axes[0].plot(path_adam[:, 0], path_adam[:, 1], 'g-', linewidth=1.5, alpha=0.7, label='Adam')
axes[0].scatter([1], [1], color='gold', s=200, marker='*', zorder=5, label='Global min (1,1)')
axes[0].scatter([x0[0]], [x0[1]], color='black', s=100, marker='o', zorder=5, label='Start')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Optimization Paths on Rosenbrock Function\n(Narrow curved valley — hard for vanilla SGD)')
axes[0].legend(loc='upper left')

# Loss curves
losses_sgd = [loss_surface(p[0], p[1]) for p in path_sgd[::10]]
losses_mom = [loss_surface(p[0], p[1]) for p in path_mom[::10]]
losses_adam = [loss_surface(p[0], p[1]) for p in path_adam[::10]]

axes[1].plot(range(len(losses_sgd)), losses_sgd, 'r-', linewidth=2, label='Vanilla SGD')
axes[1].plot(range(len(losses_mom)), losses_mom, 'b-', linewidth=2, label='Momentum')
axes[1].plot(range(len(losses_adam)), losses_adam, 'g-', linewidth=2, label='Adam')
axes[1].set_xlabel('Iteration (×10)')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Convergence Comparison')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nFinal loss values:")
print(f"  SGD:      {loss_surface(path_sgd[-1][0], path_sgd[-1][1]):.6f}")
print(f"  Momentum: {loss_surface(path_mom[-1][0], path_mom[-1][1]):.6f}")
print(f"  Adam:     {loss_surface(path_adam[-1][0], path_adam[-1][1]):.6f}")
print(f"\n🎯 Interview insights:")
print("  • Adam converges fastest (adaptive per-parameter LR)")
print("  • Momentum helps cross flat regions and narrow valleys")
print("  • SGD may generalize better in practice (noisy but explores more)")
print("  • In deep learning: Adam for prototyping, SGD+momentum for final training")

## 4.3 Backpropagation: Chain Rule in Action

### A Simple Neural Network (1 hidden layer)

Forward pass:
$$z^{(1)} = W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$$
$$\mathbf{a}^{(1)} = \sigma(z^{(1)})$$
$$z^{(2)} = W^{(2)}\mathbf{a}^{(1)} + \mathbf{b}^{(2)}$$
$$\hat{y} = \sigma(z^{(2)})$$
$$\mathcal{L} = -[y \log \hat{y} + (1-y)\log(1-\hat{y})]$$

### Backward Pass (Chain Rule)

**Output layer:**
$$\frac{\partial \mathcal{L}}{\partial z^{(2)}} = \hat{y} - y$$

$$\frac{\partial \mathcal{L}}{\partial W^{(2)}} = (\hat{y} - y) \cdot \mathbf{a}^{(1)T}$$

**Hidden layer:**
$$\frac{\partial \mathcal{L}}{\partial \mathbf{a}^{(1)}} = W^{(2)T} (\hat{y} - y)$$

$$\frac{\partial \mathcal{L}}{\partial z^{(1)}} = \frac{\partial \mathcal{L}}{\partial \mathbf{a}^{(1)}} \odot \sigma'(z^{(1)})$$

$$\frac{\partial \mathcal{L}}{\partial W^{(1)}} = \frac{\partial \mathcal{L}}{\partial z^{(1)}} \cdot \mathbf{x}^T$$

### The Vanishing/Exploding Gradient Problem

For an L-layer network, the gradient at layer 1 involves:
$$\frac{\partial \mathcal{L}}{\partial W^{(1)}} \propto \prod_{l=2}^{L} W^{(l)} \cdot \sigma'(z^{(l)})$$

- If $$\|W^{(l)}\| < 1$$ and $$\sigma' < 1$$: product shrinks exponentially → **vanishing gradients**
- If $$\|W^{(l)}\| > 1$$: product grows exponentially → **exploding gradients**

**Solutions:**
- ReLU activation ($$\sigma'(z) = 1$$ for $$z > 0$$)
- Careful initialization (Xavier/He)
- Residual connections (gradient highways)
- Gradient clipping (cap the gradient norm)
- Batch normalization

In [0]:
# === BACKPROPAGATION FROM SCRATCH ===
# Training a neural network on XOR (the classic non-linear problem)

print("="*60)
print("BACKPROPAGATION: NEURAL NET ON XOR PROBLEM")
print("="*60)
print("\nXOR is not linearly separable → need at least 1 hidden layer!")

# XOR data
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

# Activation functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

# Neural network: 2 inputs -> 4 hidden -> 1 output
class SimpleNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        # Xavier initialization
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
    
    def forward(self, X):
        """Forward pass - store intermediates for backprop."""
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.tanh(self.z1)  # tanh activation
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)  # sigmoid output
        return self.a2
    
    def backward(self, X, y, output):
        """Backward pass - compute gradients using chain rule."""
        m = X.shape[0]
        
        # Output layer gradients
        # dL/dz2 = (a2 - y) for binary cross-entropy + sigmoid
        dz2 = output - y  # Shape: (4, 1)
        dW2 = (1/m) * self.a1.T @ dz2
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        
        # Hidden layer gradients (chain rule!)
        da1 = dz2 @ self.W2.T
        dz1 = da1 * (1 - self.a1**2)  # tanh derivative: 1 - tanh^2
        dW1 = (1/m) * X.T @ dz1
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        
        return dW1, db1, dW2, db2
    
    def train(self, X, y, lr=0.5, epochs=5000):
        losses = []
        for epoch in range(epochs):
            # Forward
            output = self.forward(X)
            
            # Loss (binary cross-entropy)
            loss = -np.mean(y * np.log(output + 1e-8) + (1-y) * np.log(1 - output + 1e-8))
            losses.append(loss)
            
            # Backward
            dW1, db1, dW2, db2 = self.backward(X, y, output)
            
            # Update
            self.W1 -= lr * dW1
            self.b1 -= lr * db1
            self.W2 -= lr * dW2
            self.b2 -= lr * db2
        
        return losses

# Train the network
np.random.seed(42)
nn = SimpleNeuralNetwork(input_size=2, hidden_size=4, output_size=1)
losses = nn.train(X_xor, y_xor, lr=1.0, epochs=5000)

# Test
predictions = nn.forward(X_xor)
print(f"\nAfter training:")
print(f"  Input [0,0] → {predictions[0,0]:.4f} (target: 0)")
print(f"  Input [0,1] → {predictions[1,0]:.4f} (target: 1)")
print(f"  Input [1,0] → {predictions[2,0]:.4f} (target: 1)")
print(f"  Input [1,1] → {predictions[3,0]:.4f} (target: 0)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(losses, linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('Training Loss (Backpropagation in Action)')
axes[0].set_yscale('log')

# Decision boundary
x_min, x_max = -0.5, 1.5
y_min, y_max = -0.5, 1.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.column_stack([xx.ravel(), yy.ravel()])
zz = nn.forward(grid).reshape(xx.shape)

axes[1].contourf(xx, yy, zz, levels=50, cmap='RdYlBu_r', alpha=0.8)
axes[1].scatter(X_xor[y_xor.ravel()==0, 0], X_xor[y_xor.ravel()==0, 1], 
               c='blue', s=200, edgecolors='black', label='Class 0', zorder=5)
axes[1].scatter(X_xor[y_xor.ravel()==1, 0], X_xor[y_xor.ravel()==1, 1], 
               c='red', s=200, edgecolors='black', label='Class 1', zorder=5)
axes[1].set_title('Learned Decision Boundary for XOR\n(Non-linear — impossible with single perceptron!)')
axes[1].legend()
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

print("\n🎯 Key backprop insights for interviews:")
print("   1. Forward pass: compute output (store intermediates)")
print("   2. Backward pass: chain rule from loss back to each weight")
print("   3. Each layer's gradient depends on the layer ABOVE it")
print("   4. Computational cost: O(forward pass) — same order!")

---
# Part 5: Information Theory

Information theory, developed by Claude Shannon, provides the mathematical framework for measuring **uncertainty** and **information content**. It directly connects to loss functions in machine learning.

---

## 5.1 Entropy: Measuring Uncertainty

### Shannon Entropy

For a discrete random variable $$X$$ with PMF $$P(X=x_i) = p_i$$:

$$H(X) = -\sum_{i=1}^n p_i \log_2 p_i$$

**Interpretation:** The average number of bits needed to encode outcomes of $$X$$.

### Key Properties

1. $$H(X) \geq 0$$ (entropy is non-negative)
2. $$H(X) = 0$$ iff $$X$$ is deterministic (no uncertainty)
3. $$H(X)$$ is maximized when $$X$$ is uniform: $$H = \log_2(n)$$
4. For a coin: $$H = -p\log p - (1-p)\log(1-p)$$, maximized at $$p = 0.5$$

### Entropy in Decision Trees

**Information Gain** = reduction in entropy after splitting:
$$IG(S, A) = H(S) - \sum_{v \in A} \frac{|S_v|}{|S|} H(S_v)$$

We split on the feature that maximizes information gain (ID3 algorithm).

---

## 5.2 KL Divergence: Measuring Distribution Difference

$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)} = E_P\left[\log \frac{P(x)}{Q(x)}\right]$$

### Properties
- $$D_{KL}(P \| Q) \geq 0$$ (Gibbs' inequality)
- $$D_{KL}(P \| Q) = 0$$ iff $$P = Q$$
- **Not symmetric**: $$D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$$ (not a true distance!)

### Interpretation
- "How many extra bits do we need if we encode data from $$P$$ using a code optimized for $$Q$$?"
- Measures the "information lost" when $$Q$$ is used to approximate $$P$$

### KL Divergence in Machine Learning
- **Variational Autoencoders (VAE)**: Minimize $$D_{KL}(q(z|x) \| p(z))$$ to keep encoder close to prior
- **Policy gradient (RL)**: Trust region uses KL to limit policy updates
- **Knowledge distillation**: Teacher-student divergence

---

## 5.3 Cross-Entropy: The Loss Function Connection

$$H(P, Q) = -\sum_x P(x) \log Q(x)$$

### The Fundamental Identity

$$H(P, Q) = H(P) + D_{KL}(P \| Q)$$

Since $$H(P)$$ is constant w.r.t. model parameters:
$$\arg\min_Q H(P, Q) = \arg\min_Q D_{KL}(P \| Q)$$

**Minimizing cross-entropy = Minimizing KL divergence = Maximum Likelihood Estimation!**

This is why cross-entropy loss works:
- For classification: $$\mathcal{L} = -\sum_{c=1}^C y_c \log \hat{y}_c$$ (where $$y$$ is one-hot)
- For binary: $$\mathcal{L} = -[y \log \hat{y} + (1-y) \log(1-\hat{y})]$$

---

## 5.4 Mutual Information

$$I(X; Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = D_{KL}(P(X,Y) \| P(X)P(Y))$$

**Interpretation:** How much knowing $$Y$$ reduces uncertainty about $$X$$.

- $$I(X; Y) = 0$$ iff $$X$$ and $$Y$$ are independent
- Used in feature selection: pick features with highest MI with the target
- Captures **non-linear** relationships (unlike correlation)

In [0]:
# === INFORMATION THEORY IN ACTION ===
# Connecting entropy, cross-entropy, and KL divergence to ML

print("="*60)
print("INFORMATION THEORY: FROM ENTROPY TO LOSS FUNCTIONS")
print("="*60)

# --- 1. Entropy of a coin flip ---
print("\n--- ENTROPY ---")
p_values = np.linspace(0.001, 0.999, 200)
entropy_values = -(p_values * np.log2(p_values) + (1-p_values) * np.log2(1-p_values))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(p_values, entropy_values, 'b-', linewidth=2)
axes[0, 0].axvline(0.5, color='red', linestyle='--', label='p=0.5 (max uncertainty)')
axes[0, 0].set_xlabel('P(heads)')
axes[0, 0].set_ylabel('Entropy (bits)')
axes[0, 0].set_title('Binary Entropy: H(p) = -p log p - (1-p) log(1-p)\nMaximized at p=0.5 (1 bit = maximum uncertainty)')
axes[0, 0].legend()

print(f"Entropy of fair coin (p=0.5): {-0.5*np.log2(0.5) - 0.5*np.log2(0.5):.4f} bits")
print(f"Entropy of biased coin (p=0.9): {-0.9*np.log2(0.9) - 0.1*np.log2(0.1):.4f} bits")
print(f"Entropy of certain outcome (p=1): 0 bits")

# --- 2. KL Divergence ---
print("\n--- KL DIVERGENCE ---")

# True distribution (e.g., actual class frequencies)
P = np.array([0.7, 0.2, 0.1])  # True: 70% cat, 20% dog, 10% bird
# Model predictions (different approximations)
Q_good = np.array([0.6, 0.25, 0.15])  # Good model
Q_bad = np.array([0.33, 0.33, 0.34])  # Bad model (uniform-ish)
Q_terrible = np.array([0.1, 0.1, 0.8])  # Terrible (wrong)

def kl_divergence(P, Q):
    """KL(P || Q) = sum P(x) * log(P(x)/Q(x))"""
    return np.sum(P * np.log(P / Q))

def cross_entropy(P, Q):
    """H(P, Q) = -sum P(x) * log(Q(x))"""
    return -np.sum(P * np.log(Q))

def entropy(P):
    """H(P) = -sum P(x) * log(P(x))"""
    return -np.sum(P * np.log(P))

print(f"\nTrue distribution P = {P}")
print(f"Entropy H(P) = {entropy(P):.4f} nats")
print(f"\nModel comparisons:")
for name, Q in [('Good model', Q_good), ('Bad model', Q_bad), ('Terrible model', Q_terrible)]:
    print(f"  {name} Q={Q}:")
    print(f"    KL(P||Q) = {kl_divergence(P, Q):.4f}")
    print(f"    H(P,Q)   = {cross_entropy(P, Q):.4f}")
    print(f"    Verify:  H(P) + KL(P||Q) = {entropy(P) + kl_divergence(P, Q):.4f} = H(P,Q) ✅")

# Visualize KL divergence asymmetry
print(f"\n  KL(P||Q_good) = {kl_divergence(P, Q_good):.4f}")
print(f"  KL(Q_good||P) = {kl_divergence(Q_good, P):.4f} ≠ KL(P||Q)!")
print(f"  ⚠️ KL is NOT symmetric!")

# --- 3. Cross-entropy as loss function ---
print("\n--- CROSS-ENTROPY AS CLASSIFICATION LOSS ---")

# Binary classification example
y_true = np.array([1, 1, 0, 0, 1])  # True labels
y_pred_good = np.array([0.9, 0.8, 0.1, 0.2, 0.7])  # Good predictions
y_pred_bad = np.array([0.6, 0.4, 0.4, 0.6, 0.5])   # Bad predictions

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-10
    return -np.mean(y_true * np.log(y_pred + eps) + (1-y_true) * np.log(1-y_pred + eps))

print(f"\nBinary Cross-Entropy Loss:")
print(f"  Good predictions: {binary_cross_entropy(y_true, y_pred_good):.4f}")
print(f"  Bad predictions:  {binary_cross_entropy(y_true, y_pred_bad):.4f}")
print(f"  Perfect:          {binary_cross_entropy(y_true, np.array([1,1,0,0,1]).astype(float)+1e-10):.4f}")

# --- 4. Decision tree information gain ---
axes[0, 1].bar(['Good\nQ=[0.6,0.25,0.15]', 'Bad\nQ=[0.33,0.33,0.34]', 'Terrible\nQ=[0.1,0.1,0.8]'],
              [kl_divergence(P, Q_good), kl_divergence(P, Q_bad), kl_divergence(P, Q_terrible)],
              color=['green', 'orange', 'red'], alpha=0.7)
axes[0, 1].set_ylabel('KL Divergence (nats)')
axes[0, 1].set_title('KL Divergence: How Wrong is the Model?\n(Lower = better approximation of true P)')

# Information gain example for decision tree
axes[1, 0].set_title('Information Gain in Decision Trees\n(Split that reduces entropy most)')

# Parent node: 60% positive, 40% negative
p_parent = 0.6
H_parent = -(p_parent*np.log2(p_parent) + (1-p_parent)*np.log2(1-p_parent))

# Split A: Left (80% pos), Right (30% pos), 50-50 split
H_left_A = -(0.8*np.log2(0.8) + 0.2*np.log2(0.2))
H_right_A = -(0.3*np.log2(0.3) + 0.7*np.log2(0.7))
H_after_A = 0.5 * H_left_A + 0.5 * H_right_A
IG_A = H_parent - H_after_A

# Split B: Left (65% pos), Right (55% pos), 50-50 split
H_left_B = -(0.65*np.log2(0.65) + 0.35*np.log2(0.35))
H_right_B = -(0.55*np.log2(0.55) + 0.45*np.log2(0.45))
H_after_B = 0.5 * H_left_B + 0.5 * H_right_B
IG_B = H_parent - H_after_B

bars = axes[1, 0].bar(['Split A\n(L:80/20, R:30/70)', 'Split B\n(L:65/35, R:55/45)'], 
                      [IG_A, IG_B], color=['green', 'orange'], alpha=0.7)
axes[1, 0].set_ylabel('Information Gain (bits)')
axes[1, 0].axhline(0, color='black', linewidth=0.5)
for bar, ig in zip(bars, [IG_A, IG_B]):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
                   f'IG={ig:.4f}', ha='center', fontsize=11)

# Mutual information visualization
axes[1, 1].set_title('Mutual Information: Venn Diagram View')
from matplotlib.patches import Circle
circle1 = Circle((0.35, 0.5), 0.3, fill=False, color='blue', linewidth=2, label='H(X)')
circle2 = Circle((0.65, 0.5), 0.3, fill=False, color='red', linewidth=2, label='H(Y)')
axes[1, 1].add_patch(circle1)
axes[1, 1].add_patch(circle2)
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].text(0.2, 0.5, 'H(X|Y)', fontsize=12, ha='center', color='blue')
axes[1, 1].text(0.5, 0.5, 'I(X;Y)', fontsize=12, ha='center', color='purple', fontweight='bold')
axes[1, 1].text(0.8, 0.5, 'H(Y|X)', fontsize=12, ha='center', color='red')
axes[1, 1].text(0.5, 0.15, 'H(X,Y) = H(X) + H(Y) - I(X;Y)', fontsize=11, ha='center')
axes[1, 1].set_aspect('equal')
axes[1, 1].axis('off')
axes[1, 1].legend(loc='upper left')

plt.tight_layout()
plt.show()

print("\n🎯 Interview summary:")
print("   • Entropy: uncertainty/randomness in a distribution")
print("   • Cross-entropy loss = Entropy + KL divergence")
print("   • Minimizing CE = Minimizing KL = MLE (all the same!)")
print("   • Decision trees use Information Gain = reduction in entropy")
print("   • Mutual Information captures non-linear dependencies (unlike correlation)")

---
# Part 6: Sampling & Monte Carlo Methods

When exact computation is intractable (which is most of the time in real ML), we **approximate** using random samples.

---

## 6.1 The Bootstrap: Uncertainty from a Single Dataset

### The Problem
You have one dataset. How do you estimate the **uncertainty** of your statistic (mean, median, model accuracy)?

### The Bootstrap Principle
1. Resample $$n$$ observations **with replacement** from your dataset
2. Compute your statistic on the resample
3. Repeat B times (typically B = 1000-10000)
4. The distribution of bootstrap statistics approximates the sampling distribution

### Bootstrap Confidence Interval

**Percentile method:** Use the $$\alpha/2$$ and $$1-\alpha/2$$ percentiles of bootstrap distribution.

**BCa (Bias-Corrected and Accelerated):** Adjusts for bias and skewness — preferred in practice.

### Why Bootstrap Works

The empirical distribution function $$\hat{F}_n$$ converges to the true distribution $$F$$ (by the Glivenko-Cantelli theorem). Resampling from $$\hat{F}_n$$ simulates sampling from $$F$$.

### When Bootstrap Fails
- Extreme values (max, min) — bounded statistics
- Very small samples (n < 15-20)
- Heavy-tailed distributions without finite variance
- Dependent data (need block bootstrap)

---

## 6.2 Monte Carlo Integration

To estimate $$E[f(X)] = \int f(x) p(x) dx$$:

1. Sample $$x_1, \ldots, x_N \sim p(x)$$
2. Estimate: $$\hat{\mu} = \frac{1}{N} \sum_{i=1}^N f(x_i)$$

By LLN: $$\hat{\mu} \xrightarrow{a.s.} E[f(X)]$$

Error rate: $$O(1/\sqrt{N})$$ regardless of dimension! (Unlike grid methods which suffer from curse of dimensionality)

---

## 6.3 Markov Chain Monte Carlo (MCMC)

### The Problem

We want to sample from a complex posterior $$P(\theta | D)$$ that we can only evaluate up to a normalizing constant.

### Metropolis-Hastings Algorithm

1. Start at some $$\theta_0$$
2. Propose $$\theta' \sim Q(\theta' | \theta_t)$$ (e.g., Gaussian centered at current point)
3. Accept with probability:
$$\alpha = \min\left(1, \frac{P(\theta')Q(\theta_t|\theta')}{P(\theta_t)Q(\theta'|\theta_t)}\right)$$
4. If accepted: $$\theta_{t+1} = \theta'$$, else $$\theta_{t+1} = \theta_t$$

### Key Properties
- Generates a **Markov chain** whose stationary distribution is the target
- Need to discard initial samples ("burn-in")
- Samples are correlated (use thinning or check effective sample size)
- Acceptance rate ~23% is optimal for high dimensions (Roberts et al.)

### MCMC in Practice
- **Bayesian inference**: Sample from posterior when conjugate priors don't exist
- **Probabilistic programming**: PyMC, Stan use advanced MCMC (NUTS/HMC)
- **Sampling complex distributions**: Mixture models, hierarchical models

In [0]:
# === BOOTSTRAP AND MCMC ===
# Real-world applications of sampling methods

print("="*60)
print("BOOTSTRAP: CONFIDENCE INTERVALS WITHOUT FORMULAS")
print("="*60)

# Scenario: Estimating median income from a sample
np.random.seed(42)
# True population: right-skewed income distribution (log-normal)
true_median_income = np.exp(10.5)  # ~36,000
sample_incomes = np.random.lognormal(mean=10.5, sigma=0.8, size=50)

print(f"Sample size: {len(sample_incomes)}")
print(f"Sample median: ₹{np.median(sample_incomes):,.0f}")
print(f"True population median: ₹{true_median_income:,.0f}")

# Bootstrap
n_bootstrap = 10000
bootstrap_medians = np.array([
    np.median(np.random.choice(sample_incomes, size=len(sample_incomes), replace=True))
    for _ in range(n_bootstrap)
])

# Confidence intervals
ci_lower = np.percentile(bootstrap_medians, 2.5)
ci_upper = np.percentile(bootstrap_medians, 97.5)
bootstrap_se = bootstrap_medians.std()

print(f"\nBootstrap results ({n_bootstrap:,} resamples):")
print(f"  Bootstrap SE: ₹{bootstrap_se:,.0f}")
print(f"  95% CI (percentile): [₹{ci_lower:,.0f}, ₹{ci_upper:,.0f}]")
print(f"  True median in CI? {ci_lower <= true_median_income <= ci_upper} ✅")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bootstrap distribution
axes[0].hist(bootstrap_medians, bins=50, density=True, alpha=0.7, color='steelblue')
axes[0].axvline(np.median(sample_incomes), color='red', linewidth=2, linestyle='-', label='Sample median')
axes[0].axvline(ci_lower, color='orange', linewidth=2, linestyle='--', label=f'95% CI')
axes[0].axvline(ci_upper, color='orange', linewidth=2, linestyle='--')
axes[0].axvline(true_median_income, color='green', linewidth=2, linestyle=':', label='True median')
axes[0].set_xlabel('Median Income (₹)')
axes[0].set_title('Bootstrap Distribution of Median\n(10,000 resamples)')
axes[0].legend()

# --- MCMC: Metropolis-Hastings ---
print("\n" + "="*60)
print("MCMC: SAMPLING FROM A MIXTURE OF GAUSSIANS")
print("="*60)

# Target distribution: mixture of two Gaussians (bimodal)
def target_log_prob(x):
    """Log probability of a mixture of Gaussians."""
    return np.log(0.3 * stats.norm.pdf(x, -2, 0.5) + 0.7 * stats.norm.pdf(x, 3, 1.0))

# Metropolis-Hastings
def metropolis_hastings(target_log_prob, n_samples=50000, proposal_std=1.0, x0=0):
    samples = [x0]
    accepted = 0
    x_current = x0
    
    for _ in range(n_samples):
        # Propose
        x_proposed = x_current + np.random.normal(0, proposal_std)
        
        # Acceptance ratio (in log space for numerical stability)
        log_alpha = target_log_prob(x_proposed) - target_log_prob(x_current)
        
        # Accept/reject
        if np.log(np.random.random()) < log_alpha:
            x_current = x_proposed
            accepted += 1
        
        samples.append(x_current)
    
    return np.array(samples), accepted / n_samples

# Run MCMC with different proposal widths
for proposal_std, color, ax_idx in [(0.1, 'red', 1), (1.0, 'blue', 1), (10.0, 'green', 1)]:
    samples, acc_rate = metropolis_hastings(target_log_prob, n_samples=30000, proposal_std=proposal_std)
    burn_in = 1000
    samples = samples[burn_in:]  # Remove burn-in
    
    axes[1].hist(samples, bins=100, density=True, alpha=0.3, color=color, 
                label=f'σ={proposal_std} (acc={acc_rate:.0%})')

# True distribution
x_plot = np.linspace(-5, 7, 200)
true_pdf = 0.3 * stats.norm.pdf(x_plot, -2, 0.5) + 0.7 * stats.norm.pdf(x_plot, 3, 1.0)
axes[1].plot(x_plot, true_pdf, 'k-', linewidth=3, label='True target')
axes[1].set_title('Metropolis-Hastings MCMC\n(Effect of proposal width)')
axes[1].set_xlabel('x')
axes[1].legend(fontsize=9)

# Trace plot showing convergence
samples_trace, _ = metropolis_hastings(target_log_prob, n_samples=5000, proposal_std=1.0)
axes[2].plot(samples_trace[:2000], 'b-', alpha=0.7, linewidth=0.5)
axes[2].axhline(-2, color='red', linestyle='--', alpha=0.5, label='Mode 1 (-2)')
axes[2].axhline(3, color='green', linestyle='--', alpha=0.5, label='Mode 2 (3)')
axes[2].axvspan(0, 200, color='yellow', alpha=0.2, label='Burn-in period')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Sample value')
axes[2].set_title('MCMC Trace Plot\n(Chain explores both modes)')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n🎯 Interview insights on MCMC:")
print("   • Too narrow proposal (σ=0.1): high acceptance but slow mixing")
print("   • Too wide proposal (σ=10): low acceptance, gets stuck")
print("   • Just right (σ=1.0): ~23-44% acceptance rate is optimal")
print("   • Always discard burn-in samples and check trace plots!")

---
# Part 7: Advanced Topics

---

## 7.1 The Bias-Variance Tradeoff (Mathematical Derivation)

### Setup

True relationship: $$y = f(x) + \epsilon$$ where $$\epsilon \sim N(0, \sigma^2)$$

Model trained on a particular dataset: $$\hat{f}(x)$$

### Decomposition of Expected Test Error

For a fixed test point $$x_0$$:

$$E[(y - \hat{f}(x_0))^2] = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

$$= \underbrace{[f(x_0) - E[\hat{f}(x_0)]]^2}_{\text{Bias}^2} + \underbrace{E[(\hat{f}(x_0) - E[\hat{f}(x_0)])^2]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{Noise}}$$

### Proof

$$E[(y - \hat{f})^2] = E[(f + \epsilon - \hat{f})^2]$$
$$= E[(f - \hat{f})^2] + 2E[(f-\hat{f})\epsilon] + E[\epsilon^2]$$
$$= E[(f - \hat{f})^2] + \sigma^2$$

Now expand $$E[(f - \hat{f})^2]$$ by adding and subtracting $$E[\hat{f}]$$:
$$= E[(f - E[\hat{f}] + E[\hat{f}] - \hat{f})^2]$$
$$= (f - E[\hat{f}])^2 + E[(\hat{f} - E[\hat{f}])^2] + 2(f-E[\hat{f}])E[\hat{f}-E[\hat{f}]]$$
$$= \text{Bias}^2 + \text{Variance} + 0$$

### Implications

| | Low Bias | High Bias |
|---|---|---|
| **Low Variance** | 🎯 Ideal (impossible without enough data) | Underfitting |
| **High Variance** | Overfitting | Worst of both worlds |

- **Simple models** (linear regression): High bias, low variance
- **Complex models** (deep neural nets): Low bias, high variance
- **Regularization**: Increases bias slightly, decreases variance a lot
- **Ensemble methods**: Reduce variance (bagging) or bias (boosting)

---

## 7.2 The Kernel Trick

### Motivation

Many algorithms (SVM, regression) only use data through **dot products** $$\langle x_i, x_j \rangle$$.

### The Trick

Instead of explicitly mapping to high-dimensional space $$\phi(x)$$, define a **kernel function**:

$$K(x_i, x_j) = \langle \phi(x_i), \phi(x_j) \rangle$$

This computes the dot product in the high-dimensional space **without ever computing** $$\phi(x)$$!

### Common Kernels

| Kernel | Formula | Feature Space |
|--------|---------|---------------|
| Linear | $$K(x,y) = x^T y$$ | Same space |
| Polynomial | $$K(x,y) = (x^T y + c)^d$$ | All monomials up to degree d |
| RBF (Gaussian) | $$K(x,y) = \exp(-\gamma \|x-y\|^2)$$ | **Infinite** dimensional! |

### Why RBF Works in Infinite Dimensions

Using Taylor expansion: $$e^{-\gamma\|x-y\|^2} = \sum_{k=0}^{\infty} \frac{(-\gamma)^k}{k!} \|x-y\|^{2k}$$

This is equivalent to a dot product in an infinite-dimensional feature space, yet it's computed in $$O(d)$$ time!

---

## 7.3 Matrix Calculus Essentials

### Key Identities for ML

$$\frac{\partial}{\partial \mathbf{x}} (\mathbf{a}^T \mathbf{x}) = \mathbf{a}$$

$$\frac{\partial}{\partial \mathbf{x}} (\mathbf{x}^T A \mathbf{x}) = (A + A^T)\mathbf{x} = 2A\mathbf{x} \text{ (if A symmetric)}$$

$$\frac{\partial}{\partial X} \text{tr}(AX) = A^T$$

$$\frac{\partial}{\partial X} \text{tr}(X^T A X) = (A + A^T)X$$

### Deriving Linear Regression (OLS) Solution

Loss: $$\mathcal{L} = \|\mathbf{y} - X\mathbf{w}\|^2 = (\mathbf{y} - X\mathbf{w})^T(\mathbf{y} - X\mathbf{w})$$

Expand: $$\mathcal{L} = \mathbf{y}^T\mathbf{y} - 2\mathbf{y}^TX\mathbf{w} + \mathbf{w}^TX^TX\mathbf{w}$$

Take gradient: $$\nabla_\mathbf{w} \mathcal{L} = -2X^T\mathbf{y} + 2X^TX\mathbf{w}$$

Set to zero: $$X^TX\mathbf{w} = X^T\mathbf{y}$$

$$\boxed{\hat{\mathbf{w}} = (X^TX)^{-1}X^T\mathbf{y}}$$

This is the famous **normal equation**!

In [0]:
# === BIAS-VARIANCE TRADEOFF: VISUAL DEMONSTRATION ===
# Real-world: Choosing model complexity (polynomial degree)

print("="*60)
print("BIAS-VARIANCE TRADEOFF: POLYNOMIAL REGRESSION")
print("="*60)

# True function (unknown in practice)
def true_function(x):
    return np.sin(2 * x) + 0.5 * np.cos(4 * x)

# Generate training data (noisy observations of true function)
np.random.seed(42)
noise_std = 0.3
n_points = 25

X_train = np.sort(np.random.uniform(0, 2*np.pi, n_points))
y_train = true_function(X_train) + np.random.normal(0, noise_std, n_points)

X_test = np.linspace(0, 2*np.pi, 200)
y_test_true = true_function(X_test)

# Fit polynomials of different degrees
degrees = [1, 3, 5, 9, 15, 20]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, degree in enumerate(degrees):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]
    
    # Fit polynomial
    coeffs = np.polyfit(X_train, y_train, degree)
    y_pred = np.polyval(coeffs, X_test)
    y_pred_train = np.polyval(coeffs, X_train)
    
    # Compute errors
    train_mse = np.mean((y_train - y_pred_train)**2)
    test_mse = np.mean((y_test_true - y_pred)**2)
    
    # Plot
    ax.scatter(X_train, y_train, c='blue', s=30, alpha=0.6, label='Training data')
    ax.plot(X_test, y_test_true, 'g--', linewidth=2, label='True function')
    ax.plot(X_test, y_pred, 'r-', linewidth=2, label=f'Degree {degree} fit')
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f'Degree {degree}\nTrain MSE: {train_mse:.3f} | Test MSE: {test_mse:.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Bias-Variance Tradeoff: Underfitting → Just Right → Overfitting', fontsize=14)
plt.tight_layout()
plt.show()

# --- Quantitative bias-variance decomposition ---
print("\n" + "="*60)
print("QUANTITATIVE BIAS-VARIANCE DECOMPOSITION")
print("="*60)

n_experiments = 200
degrees_range = range(1, 20)

bias_squared_list = []
variance_list = []
total_error_list = []

for degree in degrees_range:
    predictions = np.zeros((n_experiments, len(X_test)))
    
    for exp in range(n_experiments):
        # Generate new training data each time
        X_exp = np.sort(np.random.uniform(0, 2*np.pi, n_points))
        y_exp = true_function(X_exp) + np.random.normal(0, noise_std, n_points)
        
        # Fit and predict
        try:
            coeffs = np.polyfit(X_exp, y_exp, degree)
            predictions[exp] = np.polyval(coeffs, X_test)
        except:
            predictions[exp] = np.nan
    
    # Remove nan experiments
    valid = ~np.any(np.isnan(predictions), axis=1)
    predictions = predictions[valid]
    
    if len(predictions) > 10:
        # Bias^2 = (E[f_hat] - f_true)^2
        mean_prediction = predictions.mean(axis=0)
        bias_sq = np.mean((mean_prediction - y_test_true)**2)
        
        # Variance = E[(f_hat - E[f_hat])^2]
        variance = np.mean(predictions.var(axis=0))
        
        bias_squared_list.append(bias_sq)
        variance_list.append(variance)
        total_error_list.append(bias_sq + variance + noise_std**2)
    else:
        bias_squared_list.append(np.nan)
        variance_list.append(np.nan)
        total_error_list.append(np.nan)

# Plot decomposition
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(list(degrees_range), bias_squared_list, 'b-o', linewidth=2, markersize=6, label='Bias²')
ax.plot(list(degrees_range), variance_list, 'r-s', linewidth=2, markersize=6, label='Variance')
ax.plot(list(degrees_range), total_error_list, 'k-^', linewidth=2, markersize=6, label='Total Error (Bias²+Var+Noise)')
ax.axhline(noise_std**2, color='gray', linestyle=':', label=f'Irreducible noise (σ²={noise_std**2:.2f})')

optimal_degree = np.nanargmin(total_error_list) + 1
ax.axvline(optimal_degree, color='green', linestyle='--', alpha=0.5, label=f'Optimal complexity (degree {optimal_degree})')

ax.set_xlabel('Model Complexity (Polynomial Degree)', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.set_title('Bias-Variance Decomposition\n("The U-shaped curve you must know for interviews")', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, max([x for x in total_error_list if x is not np.nan and x < 5]))

plt.tight_layout()
plt.show()

print(f"\n🎯 Optimal model complexity: degree {optimal_degree}")
print(f"   • Degrees 1-2: HIGH BIAS (underfitting, can't capture true pattern)")
print(f"   • Degree {optimal_degree}: SWEET SPOT (lowest total error)")
print(f"   • Degrees 15+: HIGH VARIANCE (overfitting, memorizes noise)")
print(f"\n   Regularization (L1/L2) achieves the same tradeoff by constraining")
print(f"   weights instead of limiting polynomial degree.")

In [0]:
# === THE KERNEL TRICK: MAKING LINEAR METHODS NON-LINEAR ===
# Real-world: SVM classification on non-linearly separable data

print("="*60)
print("THE KERNEL TRICK: SEPARATING THE INSEPARABLE")
print("="*60)

# Generate non-linearly separable data (concentric circles)
from sklearn.datasets import make_circles, make_moons
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.3)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

datasets = [('Concentric Circles', X_circles, y_circles), 
            ('Two Moons', X_moons, y_moons)]

for row, (name, X, y) in enumerate(datasets):
    # Scale data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    kernels = [('linear', 'Linear Kernel\n(Can\'t separate!)'), 
               ('poly', 'Polynomial Kernel (d=3)\n(Implicit quadratic features)'),
               ('rbf', 'RBF Kernel (γ=2)\n(Infinite-dim feature space!)')]
    
    for col, (kernel, title) in enumerate(kernels):
        ax = axes[row, col]
        
        # Fit SVM
        gamma = 2 if kernel == 'rbf' else 'scale'
        svm = SVC(kernel=kernel, degree=3, gamma=gamma, C=1.0)
        svm.fit(X_scaled, y)
        
        # Decision boundary
        xx, yy = np.meshgrid(np.linspace(X_scaled[:,0].min()-0.5, X_scaled[:,0].max()+0.5, 200),
                            np.linspace(X_scaled[:,1].min()-0.5, X_scaled[:,1].max()+0.5, 200))
        Z = svm.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
        ax.scatter(X_scaled[y==0, 0], X_scaled[y==0, 1], c='blue', s=20, alpha=0.6)
        ax.scatter(X_scaled[y==1, 0], X_scaled[y==1, 1], c='red', s=20, alpha=0.6)
        
        accuracy = svm.score(X_scaled, y)
        ax.set_title(f'{name}\n{title}\nAccuracy: {accuracy:.0%}')
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')

plt.suptitle('The Kernel Trick: Linear Methods in High-Dimensional Spaces', fontsize=14)
plt.tight_layout()
plt.show()

# --- Demonstrate what the kernel is actually computing ---
print("\n" + "="*60)
print("WHAT THE KERNEL ACTUALLY COMPUTES")
print("="*60)

# Polynomial kernel (degree 2) on 2D data
x = np.array([3, 4])  # Point 1
y_pt = np.array([1, 2])  # Point 2

# Explicit feature map for degree-2 polynomial kernel: (x^Ty + 1)^2
# φ(x) = [x1², x2², √2*x1*x2, √2*x1, √2*x2, 1]
def phi_poly2(v):
    return np.array([v[0]**2, v[1]**2, np.sqrt(2)*v[0]*v[1], 
                     np.sqrt(2)*v[0], np.sqrt(2)*v[1], 1])

# Method 1: Explicit mapping + dot product (expensive for high dimensions)
phi_x = phi_poly2(x)
phi_y = phi_poly2(y_pt)
explicit_result = np.dot(phi_x, phi_y)

# Method 2: Kernel trick (cheap!)
kernel_result = (np.dot(x, y_pt) + 1)**2

print(f"\nPolynomial kernel (degree 2) example:")
print(f"  x = {x}, y = {y_pt}")
print(f"\n  Explicit: φ(x) = {phi_x}")
print(f"           φ(y) = {phi_y}")
print(f"           φ(x)ᵀφ(y) = {explicit_result}")
print(f"\n  Kernel:  K(x,y) = (xᵀy + 1)² = ({np.dot(x, y_pt)} + 1)² = {kernel_result}")
print(f"\n  Same result! ✅ But kernel avoids computing 6D vectors.")
print(f"  For RBF kernel: the feature space is INFINITE dimensional,")
print(f"  yet K(x,y) = exp(-γ||x-y||²) is computed in O(d) time!")

print("\n🎯 Interview question: 'Can you use the kernel trick with neural networks?'")
print("   Answer: Not directly (NN aren't purely dot-product based).")
print("   But: Neural Tangent Kernel (NTK) theory shows infinite-width")
print("   neural networks are equivalent to kernel regression!")

## 7.4 Logistic Regression: Complete Mathematical Derivation

This is one of the most commonly asked derivations in interviews. Let's build it from the ground up.

### Model

$$P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T\mathbf{x} + b)}}$$

### Why Sigmoid?

The sigmoid arises naturally from modeling the **log-odds**:
$$\log \frac{P(y=1|\mathbf{x})}{P(y=0|\mathbf{x})} = \mathbf{w}^T\mathbf{x} + b$$

Solving for $$P(y=1|\mathbf{x})$$ gives the sigmoid function.

### Loss Function (Derived from MLE)

Likelihood for a single sample:
$$P(y_i | \mathbf{x}_i) = \hat{y}_i^{y_i} (1-\hat{y}_i)^{1-y_i}$$

Negative log-likelihood (the loss):
$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^n [y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)]$$

This is the **binary cross-entropy** — derived directly from MLE!

### Gradient Derivation

Let $$z_i = \mathbf{w}^T\mathbf{x}_i + b$$ and $$\hat{y}_i = \sigma(z_i)$$.

Using $$\sigma'(z) = \sigma(z)(1-\sigma(z))$$:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = -\frac{1}{n}\sum_{i=1}^n \left(\frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i}\right) \hat{y}_i(1-\hat{y}_i) \mathbf{x}_i$$

Simplifying (the magic cancellation!):
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{n}\sum_{i=1}^n (\hat{y}_i - y_i) \mathbf{x}_i = \frac{1}{n} X^T(\hat{\mathbf{y}} - \mathbf{y})$$

**Beautiful result:** The gradient has the same form as linear regression! This is because of the choice of sigmoid + cross-entropy — they are **canonical** pairs in exponential family theory.

### Why is the Loss Convex?

The Hessian of the logistic loss is:
$$H = \frac{1}{n} X^T D X$$

where $$D = \text{diag}(\hat{y}_i(1-\hat{y}_i))$$ is diagonal with positive entries.

Since $$D \succ 0$$, we have $$H \succeq 0$$, proving the loss is convex. Gradient descent always finds the global optimum!

In [0]:
# === LOGISTIC REGRESSION: FROM MATH TO CODE ===
# Complete implementation matching the derivation above

print("="*60)
print("LOGISTIC REGRESSION FROM SCRATCH")
print("="*60)

# Real-world data: Predicting customer churn from usage patterns
np.random.seed(42)
n = 500

# Features: monthly_usage (hours), satisfaction_score
monthly_usage = np.random.normal(20, 8, n)
satisfaction = np.random.normal(7, 2, n)

# True churn probability depends on both features
z_true = -0.15 * monthly_usage + -0.8 * satisfaction + 7
churn_prob = 1 / (1 + np.exp(-z_true))
churn = (np.random.random(n) < churn_prob).astype(float)

print(f"Dataset: {n} customers")
print(f"Churn rate: {churn.mean():.1%}")
print(f"Features: monthly_usage, satisfaction_score")

# Prepare data
X = np.column_stack([monthly_usage, satisfaction])
X = (X - X.mean(axis=0)) / X.std(axis=0)  # Standardize
X = np.column_stack([np.ones(n), X])  # Add bias column
y = churn

class LogisticRegressionFromScratch:
    """Logistic regression implementing the exact math from the derivation."""
    
    def __init__(self):
        self.weights = None
        self.losses = []
    
    def sigmoid(self, z):
        """Numerically stable sigmoid."""
        return np.where(z >= 0, 
                       1 / (1 + np.exp(-z)),
                       np.exp(z) / (1 + np.exp(z)))
    
    def predict_proba(self, X):
        """P(y=1|x) = σ(w^T x)"""
        return self.sigmoid(X @ self.weights)
    
    def compute_loss(self, X, y):
        """Binary cross-entropy: L = -(1/n) Σ[y*log(ŷ) + (1-y)*log(1-ŷ)]"""
        y_hat = self.predict_proba(X)
        eps = 1e-10  # Prevent log(0)
        return -np.mean(y * np.log(y_hat + eps) + (1-y) * np.log(1-y_hat + eps))
    
    def compute_gradient(self, X, y):
        """Gradient: (1/n) X^T (ŷ - y)  [the beautiful simplified form]"""
        y_hat = self.predict_proba(X)
        return (1/len(y)) * X.T @ (y_hat - y)
    
    def fit(self, X, y, lr=0.1, n_iter=1000, verbose=True):
        """Train using gradient descent."""
        self.weights = np.zeros(X.shape[1])
        self.losses = []
        
        for i in range(n_iter):
            loss = self.compute_loss(X, y)
            self.losses.append(loss)
            
            gradient = self.compute_gradient(X, y)
            self.weights -= lr * gradient
            
            if verbose and (i % 200 == 0 or i == n_iter-1):
                acc = ((self.predict_proba(X) > 0.5) == y).mean()
                print(f"  Iter {i:4d} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")
        
        return self

# Train
model = LogisticRegressionFromScratch()
print("\nTraining:")
model.fit(X, y, lr=0.5, n_iter=1000)

# Final results
y_pred_prob = model.predict_proba(X)
y_pred = (y_pred_prob > 0.5).astype(int)
final_accuracy = (y_pred == y).mean()

print(f"\nFinal weights: bias={model.weights[0]:.4f}, usage={model.weights[1]:.4f}, satisfaction={model.weights[2]:.4f}")
print(f"Final accuracy: {final_accuracy:.1%}")

# Verify gradient numerically
print("\n--- GRADIENT VERIFICATION (numerical vs analytical) ---")
eps = 1e-5
numerical_grad = np.zeros_like(model.weights)
for j in range(len(model.weights)):
    w_plus = model.weights.copy()
    w_plus[j] += eps
    w_minus = model.weights.copy()
    w_minus[j] -= eps
    
    model_plus = LogisticRegressionFromScratch()
    model_plus.weights = w_plus
    model_minus = LogisticRegressionFromScratch()
    model_minus.weights = w_minus
    
    numerical_grad[j] = (model_plus.compute_loss(X, y) - model_minus.compute_loss(X, y)) / (2*eps)

analytical_grad = model.compute_gradient(X, y)
print(f"Analytical gradient: {analytical_grad}")
print(f"Numerical gradient:  {numerical_grad}")
print(f"Max difference: {np.max(np.abs(analytical_grad - numerical_grad)):.2e} ✅")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(model.losses, linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('Convergence (Convex → guaranteed global minimum)')

# Decision boundary
xx = np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 100)
yy = np.linspace(X[:, 2].min()-0.5, X[:, 2].max()+0.5, 100)
XX, YY = np.meshgrid(xx, yy)
X_grid = np.column_stack([np.ones(XX.ravel().shape[0]), XX.ravel(), YY.ravel()])
Z = model.predict_proba(X_grid).reshape(XX.shape)

axes[1].contourf(XX, YY, Z, levels=20, cmap='RdYlBu_r', alpha=0.7)
axes[1].scatter(X[y==0, 1], X[y==0, 2], c='blue', s=20, alpha=0.5, label='No churn')
axes[1].scatter(X[y==1, 1], X[y==1, 2], c='red', s=20, alpha=0.5, label='Churned')
axes[1].set_xlabel('Monthly Usage (standardized)')
axes[1].set_ylabel('Satisfaction Score (standardized)')
axes[1].set_title('Decision Boundary\n(P(churn) = 0.5 contour)')
axes[1].legend()

# Sigmoid function
z_range = np.linspace(-6, 6, 200)
axes[2].plot(z_range, 1/(1+np.exp(-z_range)), 'b-', linewidth=3)
axes[2].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[2].axvline(0, color='gray', linestyle=':', alpha=0.5)
axes[2].set_xlabel('z = wᵀx + b')
axes[2].set_ylabel('σ(z) = P(y=1|x)')
axes[2].set_title('The Sigmoid Function\nσ(z) = 1/(1+e$^{-z}$)')
axes[2].annotate('σ\'(z) = σ(z)(1-σ(z))\nMax slope at z=0', xy=(0, 0.5), xytext=(2, 0.3),
               fontsize=11, arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

---
## 7.5 Classic Interview Problems (Worked Solutions)

### Problem 1: The Coupon Collector

**Q:** A cereal box contains one of $$n$$ different toys (uniformly random). How many boxes must you buy to collect all $$n$$ toys?

**Solution:** After collecting $$k$$ distinct toys, the probability of getting a new one is $$\frac{n-k}{n}$$.

Time to get the next new toy: $$\text{Geometric}\left(\frac{n-k}{n}\right)$$ with mean $$\frac{n}{n-k}$$.

$$E[T] = \sum_{k=0}^{n-1} \frac{n}{n-k} = n \sum_{j=1}^n \frac{1}{j} = n \cdot H_n \approx n \ln n + \gamma n$$

where $$H_n$$ is the harmonic number and $$\gamma \approx 0.577$$ is Euler's constant.

---

### Problem 2: Expected Value of Max of Uniform RVs

**Q:** Draw $$n$$ samples from $$\text{Uniform}(0,1)$$. What's $$E[\max(X_1, \ldots, X_n)]$$?

**Solution:**
$$P(\max \leq x) = P(X_1 \leq x, \ldots, X_n \leq x) = x^n$$

$$f_{\max}(x) = nx^{n-1}$$

$$E[\max] = \int_0^1 x \cdot nx^{n-1} dx = n \cdot \frac{x^{n+1}}{n+1}\bigg|_0^1 = \frac{n}{n+1}$$

**Application:** In online advertising, if you show an ad to $$n$$ users, the maximum CTR you observe is biased upward by this factor. This is the "winner's curse" in A/B testing.

---

### Problem 3: Conditional Expectation (The Two Envelopes)

**Q:** You pick a random card from a shuffled deck until you get an Ace. What's the expected number of cards drawn?

**Solution:** Think of it as: where is the first Ace among all 52 cards?

The 4 Aces divide the 52 cards into 5 groups. By symmetry, the expected size of the first group (before the first Ace) is $$\frac{48}{5}$$.

So expected draws = $$\frac{48}{5} + 1 = \frac{53}{5} = 10.6$$

---

### Problem 4: Markov's and Chebyshev's Inequalities

**Markov:** For non-negative $$X$$: $$P(X \geq a) \leq \frac{E[X]}{a}$$

**Chebyshev:** $$P(|X - \mu| \geq k\sigma) \leq \frac{1}{k^2}$$

**Application:** "At most 25% of users can be more than 2 standard deviations from the mean engagement." This holds for ANY distribution!

In [0]:
# === CLASSIC INTERVIEW PROBLEMS: SIMULATIONS ===

print("="*60)
print("CLASSIC INTERVIEW PROBLEMS: THEORY MEETS SIMULATION")
print("="*60)

# --- Problem 1: Coupon Collector ---
print("\n--- COUPON COLLECTOR PROBLEM ---")

def coupon_collector_sim(n_toys, n_simulations=10000):
    """Simulate: How many boxes to collect all n toys?"""
    results = []
    for _ in range(n_simulations):
        collected = set()
        boxes = 0
        while len(collected) < n_toys:
            collected.add(np.random.randint(0, n_toys))
            boxes += 1
        results.append(boxes)
    return np.array(results)

for n in [5, 10, 50, 100]:
    simulated = coupon_collector_sim(n, 5000)
    # Theoretical: n * H_n
    H_n = sum(1/k for k in range(1, n+1))  # Harmonic number
    theoretical = n * H_n
    print(f"  n={n:3d}: Simulated E[T]={simulated.mean():.1f}, Theory n*H_n={theoretical:.1f}")

# --- Problem 2: Expected Maximum ---
print("\n--- EXPECTED VALUE OF MAX(X_1, ..., X_n) ---")
print("   X_i ~ Uniform(0,1)")

for n in [2, 5, 10, 50, 100]:
    simulated_maxes = [np.random.random(n).max() for _ in range(10000)]
    theoretical_max = n / (n + 1)
    print(f"  n={n:3d}: Simulated E[max]={np.mean(simulated_maxes):.4f}, Theory n/(n+1)={theoretical_max:.4f}")

print("\n  💡 Application: The 'winner\'s curse' in A/B testing.")
print("     If you test 20 variations, the best observed lift is biased HIGH.")
print(f"     E[max of 20 Uniform(0,1)] = {20/21:.4f} (inflated from true 0.5)")

# --- Problem 3: Chebyshev's Inequality ---
print("\n--- CHEBYSHEV'S INEQUALITY ---")
print("   P(|X - μ| ≥ kσ) ≤ 1/k²")

# Demonstrate with a highly skewed distribution (exponential)
samples = np.random.exponential(scale=1.0, size=100000)
mu = samples.mean()
sigma = samples.std()

print(f"\n  Distribution: Exponential(1), μ={mu:.3f}, σ={sigma:.3f}")
print(f"  {'k':>4} | {'Chebyshev bound':>16} | {'Actual P(|X-μ|≥kσ)':>22}")
print(f"  {'-'*4}-+-{'-'*16}-+-{'-'*22}")

for k in [1, 1.5, 2, 3, 4, 5]:
    chebyshev_bound = 1 / k**2
    actual_prob = np.mean(np.abs(samples - mu) >= k * sigma)
    print(f"  {k:4.1f} | {chebyshev_bound:16.4f} | {actual_prob:22.4f}")

print("\n  ✅ Actual probability is always ≤ Chebyshev bound!")
print("  (Bound is loose because it works for ANY distribution)")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coupon collector distribution
results_50 = coupon_collector_sim(50, 10000)
axes[0].hist(results_50, bins=50, density=True, alpha=0.7, color='steelblue')
H_50 = sum(1/k for k in range(1, 51))
axes[0].axvline(50 * H_50, color='red', linewidth=2, linestyle='--', 
               label=f'E[T] = 50×H_50 = {50*H_50:.0f}')
axes[0].set_xlabel('Boxes needed')
axes[0].set_ylabel('Density')
axes[0].set_title('Coupon Collector (n=50 toys)\nHow many boxes to get all toys?')
axes[0].legend()

# Expected max vs n
ns = np.arange(1, 101)
expected_maxes = ns / (ns + 1)
axes[1].plot(ns, expected_maxes, 'b-', linewidth=2)
axes[1].axhline(1, color='red', linestyle='--', alpha=0.5, label='Limit = 1')
axes[1].set_xlabel('n (number of samples)')
axes[1].set_ylabel('E[max(X_1, ..., X_n)]')
axes[1].set_title('Expected Maximum of n Uniform(0,1) RVs\nE[max] = n/(n+1)')
axes[1].legend()
axes[1].annotate('n=20 → E[max]=0.95\n(Winner\'s curse!)', xy=(20, 20/21), 
                xytext=(40, 0.8), fontsize=11, arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

---
# Summary: Quick Reference Card

## The "Mount Rushmore" of DS Math

### 1. The Chain Rule (Backpropagation)
$$\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

### 2. Bayes' Theorem (All of Bayesian ML)
$$P(\theta | D) = \frac{P(D | \theta) P(\theta)}{P(D)}$$

### 3. The Normal Equation (Linear Regression)
$$\hat{\mathbf{w}} = (X^TX)^{-1}X^T\mathbf{y}$$

### 4. The Cross-Entropy Identity (Loss Functions)
$$H(P, Q) = H(P) + D_{KL}(P \| Q)$$

### 5. The Bias-Variance Decomposition
$$E[(y - \hat{f})^2] = \text{Bias}^2 + \text{Variance} + \sigma^2$$

---

## Top 10 Interview Connections

| Question | Math Foundation |
|----------|----------------|
| Why does PCA work? | Eigenvectors of covariance matrix maximize variance |
| Why L1 gives sparsity? | L1 ball has corners on axes |
| Why regularization = prior? | MAP estimation with Gaussian/Laplace prior |
| Why cross-entropy loss? | MLE derivation for Bernoulli likelihood |
| Why gradient descent converges? | Convexity + Lipschitz gradients |
| Why does dropout work? | Approximate Bayesian inference / ensemble |
| Why √n in standard error? | Central Limit Theorem |
| Why bootstrap works? | Glivenko-Cantelli theorem |
| Why random forests reduce variance? | Averaging decorrelated predictors |
| Why attention scales by √d_k? | Preventing softmax saturation in high dimensions |

---

## Study Strategy

1. **Can you derive it?** If not, you'll struggle to explain it in an interview.
2. **Can you code it?** Implementation reveals gaps in understanding.
3. **Can you explain when it breaks?** Every method has failure modes. Know them.
4. **Can you connect it to practice?** "This is just X applied to Y" shows real mastery.